In [1]:
"""
MODULE 10 — FRAUD DETECTION, FINANCIAL CRIME INTELLIGENCE &
            ANOMALY MONITORING PLATFORM

Author  : Senior Fraud Analytics Architect & Financial Crime Intelligence Consultant
Depends : Modules 1–9 outputs
Primary : lending_features/master_feature_table.csv + all module outputs
Outputs : fraud_detection/  — models, networks, alerts, dashboards, reports

Steps:
  1  Fraud Intelligence Framework
  2  Fraud Taxonomy Design
  3  Fraud Feature Engineering
  4  Anomaly Detection Engine (Isolation Forest / LOF / One-Class SVM / Autoencoder)
  5  Synthetic Identity Detection
  6  Behavioral Fraud Analytics
  7  Transaction Fraud Detection
  8  Application Fraud Detection
  9  Collusive & Network Fraud Detection (NetworkX)
  10 Fraud Risk Scoring Engine
  11 Fraud Explainability Layer (SHAP)
  12 Fraud Alerting System
  13 Real-Time Fraud Monitoring Simulation
  14 Fraud Governance & Compliance
  15 Stress Testing & Adversarial Simulation
  16 Executive Fraud Intelligence Dashboards
  17 Advanced Visual Analytics
  18 Fairness & Ethical Fraud Analytics
  19 Production Fraud Detection Pipeline
"""

'\nMODULE 10 — FRAUD DETECTION, FINANCIAL CRIME INTELLIGENCE &\n            ANOMALY MONITORING PLATFORM\n\nAuthor  : Senior Fraud Analytics Architect & Financial Crime Intelligence Consultant\nDepends : Modules 1–9 outputs\nPrimary : lending_features/master_feature_table.csv + all module outputs\nOutputs : fraud_detection/  — models, networks, alerts, dashboards, reports\n\nSteps:\n  1  Fraud Intelligence Framework\n  2  Fraud Taxonomy Design\n  3  Fraud Feature Engineering\n  4  Anomaly Detection Engine (Isolation Forest / LOF / One-Class SVM / Autoencoder)\n  5  Synthetic Identity Detection\n  6  Behavioral Fraud Analytics\n  7  Transaction Fraud Detection\n  8  Application Fraud Detection\n  9  Collusive & Network Fraud Detection (NetworkX)\n  10 Fraud Risk Scoring Engine\n  11 Fraud Explainability Layer (SHAP)\n  12 Fraud Alerting System\n  13 Real-Time Fraud Monitoring Simulation\n  14 Fraud Governance & Compliance\n  15 Stress Testing & Adversarial Simulation\n  16 Executive Frau

In [2]:
# =============================================================================
# CELL 1 — Install dependencies
# =============================================================================
!pip install pandas numpy scipy scikit-learn matplotlib seaborn plotly \
             xgboost lightgbm shap networkx joblib tqdm statsmodels \
             pyod --quiet


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Users\abhir\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [3]:
# =============================================================================
# CELL 2 — Imports & Global Configuration
# =============================================================================
import os, sys, json, warnings, logging, joblib, time
from pathlib import Path
from datetime import datetime, timedelta
from copy import deepcopy
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import ks_2samp

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.neural_network import MLPClassifier

import networkx as nx

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False
    print("⚠  SHAP not found — install: pip install shap")

try:
    from xgboost import XGBClassifier
    from lightgbm import LGBMClassifier
    BOOST_OK = True
except ImportError:
    BOOST_OK = False

try:
    from pyod.models.iforest import IForest
    from pyod.models.lof    import LOF
    from pyod.models.ocsvm  import OCSVM
    from pyod.models.copod  import COPOD
    from pyod.models.hbos   import HBOS
    PYOD_OK = True
except ImportError:
    PYOD_OK = False
    print("⚠  PyOD not found — install: pip install pyod")

warnings.filterwarnings("ignore")
try:
    get_ipython()
    matplotlib.use("inline")
except NameError:
    matplotlib.use("Agg")

SEED = 42
np.random.seed(SEED)

# ── Palette ────────────────────────────────────────────────────────────────
PAL = {
    "primary":   "#2C5F8A",
    "danger":    "#D94F3D",
    "success":   "#2E8B57",
    "warning":   "#E8A838",
    "neutral":   "#6B7280",
    "highlight": "#7B2D8B",
    "orange":    "#E07B39",
    "bg":        "#F8F9FA",
    "teal":      "#1ABC9C",
    "gold":      "#C9A227",
}
FRAUD_COLORS = {
    "Green":  "#2E8B57",
    "Yellow": "#E8A838",
    "Orange": "#E07B39",
    "Red":    "#D94F3D",
}
SEVERITY_PALETTE = ["#2E8B57", "#E8A838", "#E07B39", "#D94F3D"]

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "figure.facecolor": PAL["bg"],
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.titleweight": "bold",
    "axes.titlesize":   12,
})

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR  = r"C:\Users\abhir\OneDrive\Desktop\iitg"
FEAT_DIR  = os.path.join(BASE_DIR, "lending_features")
FD_DIR    = os.path.join(BASE_DIR, "fraud_detection")
MOD_DIR   = os.path.join(FD_DIR,  "anomaly_models")
NET_DIR   = os.path.join(FD_DIR,  "fraud_networks")
XAI_DIR   = os.path.join(FD_DIR,  "explainability")
DASH_DIR  = os.path.join(FD_DIR,  "dashboards")
RPT_DIR   = os.path.join(FD_DIR,  "reports")
ALERT_DIR = os.path.join(FD_DIR,  "alerts")
GOV_DIR   = os.path.join(FD_DIR,  "governance")
STR_DIR   = os.path.join(FD_DIR,  "stress_testing")
PIP_DIR   = os.path.join(FD_DIR,  "pipelines")
NB_DIR    = os.path.join(FD_DIR,  "notebooks")

for d in [FD_DIR, MOD_DIR, NET_DIR, XAI_DIR, DASH_DIR, RPT_DIR,
           ALERT_DIR, GOV_DIR, STR_DIR, PIP_DIR, NB_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ── Logging ────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.FileHandler(
            os.path.join(FD_DIR, "module10.log"), mode="w", encoding="utf-8"
        ),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("FraudIntel")

def savefig(name, dpi=150, folder=DASH_DIR):
    path = os.path.join(folder, name)
    plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor=PAL["bg"])
    plt.close()
    log.info("  Figure: %s", name)
    return path

def section(title):
    bar = "═" * 70
    print(f"\n{bar}\n  {title}\n{bar}")

log.info("Module 10 — Fraud Detection & Financial Crime Intelligence started.")
print("✅  Module 10 configuration loaded. All fraud_detection/ dirs ready.")

11:01:12 | INFO     | Module 10 — Fraud Detection & Financial Crime Intelligence started.
✅  Module 10 configuration loaded. All fraud_detection/ dirs ready.


In [4]:
# =============================================================================
# CELL 3 — STEP 1: Fraud Intelligence Framework
# =============================================================================

section("STEP 1 — FRAUD INTELLIGENCE FRAMEWORK")

FRAUD_INTEL_FRAMEWORK = """
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 10 — FRAUD INTELLIGENCE & FINANCIAL CRIME FRAMEWORK          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHY FRAUD INTELLIGENCE IN DIGITAL LENDING?                          ║
║  Digital lending creates unique fraud attack surfaces:               ║
║  • Remote identity verification — synthetic ID creation is easy     ║
║  • Rapid application cycle — less human review per application      ║
║  • Behavioral data richness — signals detectable before loss occurs ║
║  • Network effects — fraud rings exploit approval pipelines          ║
║  • API-driven disbursement — funds moved instantly post-approval    ║
║                                                                      ║
║  FRAUD LIFECYCLE IN NBFC / FINTECH LENDING:                          ║
║  1. IDENTITY CREATION  → Synthetic borrower profiles built          ║
║  2. APPLICATION        → Fraudulent documents/income submitted      ║
║  3. UNDERWRITING BYPASS→ Score manipulation; collusion in rings     ║
║  4. DISBURSEMENT       → Funds rapidly moved (mule accounts)        ║
║  5. REPAYMENT GAMING   → Strategic payments to avoid alerts         ║
║  6. DEFAULT            → Planned default; maximum loan extracted    ║
║                                                                      ║
║  TYPES OF LENDING FRAUD:                                             ║
║  • Application Fraud   — fake income, identity, employment          ║
║  • Synthetic Identity  — manufactured personas, no real borrower    ║
║  • Account Takeover    — legitimate account hijacked                ║
║  • Repayment Fraud     — strategic payment manipulation             ║
║  • Mule Activity       — accounts used for fund laundering          ║
║  • Collusive Rings     — coordinated group exploiting limits        ║
║  • First-Party Fraud   — real borrower, intentional default         ║
║  • Third-Party Fraud   — fraudster using another person's identity  ║
║                                                                      ║
║  FRAUD-PORTFOLIO RISK INTERACTION:                                   ║
║  • Fraud NPA → misclassified credit risk → model degradation       ║
║  • Fraud clusters → correlated losses → concentration risk spike    ║
║  • Undetected fraud → write-offs misattributed to credit risk       ║
║  • Fraud rings → EWS signals contaminated → false EWS triggers     ║
║                                                                      ║
║  FRAUD DETECTION APPROACH:                                           ║
║  Layer 1: Rule-Based Heuristics  (fast, interpretable)              ║
║  Layer 2: Anomaly Detection      (unsupervised, novelty detection)  ║
║  Layer 3: Supervised ML          (labelled fraud signals)           ║
║  Layer 4: Network Analytics      (graph-based collusion detection)  ║
║  Layer 5: Real-Time Monitoring   (streaming behavioral signals)     ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(FRAUD_INTEL_FRAMEWORK)

FRAUD_GOVERNANCE_OBJECTIVES = {
    "primary_objective":    "Detect and prevent financial crime before loss occurs",
    "secondary_objectives": [
        "Protect portfolio profitability from fraud-driven write-offs",
        "Generate audit-ready fraud intelligence for compliance",
        "Support underwriting with real-time fraud risk signals",
        "Enable collections to differentiate fraud from genuine default",
        "Provide executive leadership with fraud exposure dashboards",
    ],
    "fraud_loss_targets": {
        "max_fraud_loss_pct_aum":    0.008,   # < 80 bps of AUM
        "max_synthetic_id_pct":      0.005,   # < 50 bps of applicants
        "min_fraud_detection_recall":0.85,    # catch ≥ 85% of fraud
        "max_fpr":                   0.05,    # < 5% false positives
        "max_days_to_detection":     30,      # detect within 30 days
    },
    "regulatory_alignment": [
        "RBI Master Direction on KYC",
        "PMLA 2002 — Anti-Money Laundering",
        "IT Act 2000 — Cyber fraud provisions",
        "SEBI LODR — Fraud disclosure norms",
    ]
}

with open(os.path.join(GOV_DIR, "fraud_governance_objectives.json"), "w") as f:
    json.dump(FRAUD_GOVERNANCE_OBJECTIVES, f, indent=2)
with open(os.path.join(RPT_DIR, "fraud_intel_framework.txt"), "w", encoding="utf-8") as f:
    f.write(FRAUD_INTEL_FRAMEWORK)

print("  ✅  Fraud intelligence framework saved.")
log.info("Fraud intelligence framework complete.")


══════════════════════════════════════════════════════════════════════
  STEP 1 — FRAUD INTELLIGENCE FRAMEWORK
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 10 — FRAUD INTELLIGENCE & FINANCIAL CRIME FRAMEWORK          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  WHY FRAUD INTELLIGENCE IN DIGITAL LENDING?                          ║
║  Digital lending creates unique fraud attack surfaces:               ║
║  • Remote identity verification — synthetic ID creation is easy     ║
║  • Rapid application cycle — less human review per application      ║
║  • Behavioral data richness — signals detectable before loss occurs ║
║  • Network effects — fraud rings exploit approval pipelines          ║
║  • API-driven disbursement — funds moved instantly post-approval    ║
║                

In [5]:
# =============================================================================
# CELL 4 — STEP 2: Fraud Taxonomy Design
# =============================================================================

section("STEP 2 — FRAUD TAXONOMY DESIGN")

FRAUD_TAXONOMY = {
    "APPLICATION_FRAUD": {
        "description": "Fraudulent information submitted at loan origination",
        "severity":    "HIGH",
        "subcategories": {
            "income_inflation": {
                "definition": "Declared income significantly exceeds verifiable sources",
                "signals": ["income_to_emi_ratio outlier", "employer mismatch",
                             "salary credit pattern inconsistency"],
                "detection": "Rule-based + anomaly score > P95",
            },
            "fake_employment": {
                "definition": "Employment details fabricated or unverifiable",
                "signals": ["employer not in verified registry",
                             "salary pattern not matching employment type",
                             "new employer + high income request"],
                "detection": "Identity consistency scoring",
            },
            "identity_manipulation": {
                "definition": "Personal details altered or borrowed from another person",
                "signals": ["address/DOB inconsistency",
                             "multiple applications with similar PII",
                             "device fingerprint reuse across identities"],
                "detection": "Network analysis + PII similarity scoring",
            },
            "document_forgery": {
                "definition": "Altered or fabricated financial documents submitted",
                "signals": ["document metadata mismatch",
                             "income certificate inconsistency with bank statements"],
                "detection": "Document risk scoring",
            },
        }
    },
    "SYNTHETIC_IDENTITY_FRAUD": {
        "description": "Manufactured borrower personas not linked to any real individual",
        "severity":    "CRITICAL",
        "subcategories": {
            "synthetic_pii": {
                "definition": "Combination of real and fabricated personal information",
                "signals": ["thin credit file + high loan ask",
                             "DOB in statistical anomaly range",
                             "address never appears in credit bureau"],
                "detection": "Synthetic identity probability model",
            },
            "credit_piggybacking": {
                "definition": "Authorized user added to established account to boost score",
                "signals": ["sudden credit score spike",
                             "credit age inconsistent with behavioral history"],
                "detection": "Credit trajectory anomaly detection",
            },
        }
    },
    "BEHAVIORAL_FRAUD": {
        "description": "Post-approval behavioral patterns indicating fraudulent intent",
        "severity":    "MEDIUM",
        "subcategories": {
            "rapid_cashout": {
                "definition": "Loan funds withdrawn immediately after disbursement",
                "signals": ["ATM withdrawal > 80% within 48h",
                             "multiple rapid transfers post-disbursement"],
                "detection": "Transaction velocity analysis",
            },
            "repayment_gaming": {
                "definition": "Strategic payments timed to avoid delinquency flags",
                "signals": ["payments within 1-2 days of DPD threshold",
                             "alternating payment / non-payment pattern"],
                "detection": "Repayment timing anomaly detection",
            },
            "device_account_sharing": {
                "definition": "Single device used across multiple borrower accounts",
                "signals": ["device fingerprint match across accounts",
                             "same IP for multiple distinct borrowers"],
                "detection": "Network overlap analysis",
            },
        }
    },
    "TRANSACTION_FRAUD": {
        "description": "Anomalous transaction patterns post-disbursement",
        "severity":    "HIGH",
        "subcategories": {
            "mule_activity": {
                "definition": "Account used as conduit for laundering proceeds",
                "signals": ["large inflows immediately followed by outflows",
                             "multiple senders → single receiver"],
                "detection": "Transaction graph analysis",
            },
            "circular_transfers": {
                "definition": "Money cycled through multiple accounts to obscure origin",
                "signals": ["cycle detection in transaction graph",
                             "same amount returned within days"],
                "detection": "NetworkX cycle detection",
            },
        }
    },
    "COLLUSIVE_FRAUD": {
        "description": "Coordinated groups of individuals exploiting lending systems",
        "severity":    "CRITICAL",
        "subcategories": {
            "fraud_ring": {
                "definition": "Multiple borrowers coordinated by a single operator",
                "signals": ["shared contact references",
                             "similar application timing",
                             "overlapping employer, address, device"],
                "detection": "Graph community detection",
            },
            "coordinated_default": {
                "definition": "Multiple borrowers default simultaneously without explanation",
                "signals": ["correlated delinquency timelines",
                             "no macro-stress explanation",
                             "connected in fraud network"],
                "detection": "Temporal correlation analysis + graph analytics",
            },
        }
    },
}

FRAUD_SEVERITY_HIERARCHY = {
    "CRITICAL": {"score_range": (80, 100), "action": "Immediate investigation + hold disbursement",
                  "escalation": "CRO + Legal within 24h", "color": "Red"},
    "HIGH":     {"score_range": (60, 79),  "action": "Senior analyst review within 48h",
                  "escalation": "Fraud Analytics Head", "color": "Orange"},
    "MEDIUM":   {"score_range": (40, 59),  "action": "Enhanced monitoring + soft intervention",
                  "escalation": "Fraud Analyst", "color": "Yellow"},
    "LOW":      {"score_range": (0,  39),  "action": "Automated monitoring only",
                  "escalation": "None", "color": "Green"},
}

with open(os.path.join(GOV_DIR, "fraud_taxonomy.json"), "w") as f:
    json.dump(FRAUD_TAXONOMY, f, indent=2)
with open(os.path.join(GOV_DIR, "fraud_severity_hierarchy.json"), "w") as f:
    json.dump(FRAUD_SEVERITY_HIERARCHY, f, indent=2)

print(f"  Fraud categories defined: {len(FRAUD_TAXONOMY)}")
for cat, info in FRAUD_TAXONOMY.items():
    n_sub = len(info.get("subcategories", {}))
    print(f"    {cat:<35} Severity={info['severity']:<8} SubTypes={n_sub}")

print("  ✅  Fraud taxonomy and severity hierarchy saved.")
log.info("Fraud taxonomy design complete.")


══════════════════════════════════════════════════════════════════════
  STEP 2 — FRAUD TAXONOMY DESIGN
══════════════════════════════════════════════════════════════════════
  Fraud categories defined: 5
    APPLICATION_FRAUD                   Severity=HIGH     SubTypes=4
    SYNTHETIC_IDENTITY_FRAUD            Severity=CRITICAL SubTypes=2
    BEHAVIORAL_FRAUD                    Severity=MEDIUM   SubTypes=3
    TRANSACTION_FRAUD                   Severity=HIGH     SubTypes=2
    COLLUSIVE_FRAUD                     Severity=CRITICAL SubTypes=2
  ✅  Fraud taxonomy and severity hierarchy saved.
11:03:00 | INFO     | Fraud taxonomy design complete.


In [6]:
# =============================================================================
# CELL 5 — Data Loading & Preparation
# =============================================================================

section("DATA LOADING & PORTFOLIO PREPARATION")

path = os.path.join(FEAT_DIR, "master_feature_table.csv")
if not os.path.exists(path):
    raise FileNotFoundError("master_feature_table.csv not found. Run Module 2 first.")

df_raw = pd.read_csv(path, low_memory=False)
log.info("Master table loaded: %s rows × %s cols", f"{len(df_raw):,}", df_raw.shape[1])
df = df_raw.copy()
print(f"  Master table: {df.shape}")

# Work with approved loans — fraud lives in the approved book
approved = df[df["approval_status"] == "Approved"].copy().reset_index(drop=True) \
           if "approval_status" in df.columns else df.copy()
print(f"  Approved subset: {len(approved):,} rows")

# ── Ensure core columns exist ──────────────────────────────────────────────
def ensure_col(df_in, col, fn):
    if col not in df_in.columns:
        df_in[col] = fn(df_in)
    return df_in

approved = ensure_col(approved, "credit_score",
    lambda d: np.random.randint(300, 900, len(d)))
approved = ensure_col(approved, "monthly_income",
    lambda d: np.random.uniform(15000, 120000, len(d)))
approved = ensure_col(approved, "loan_amount",
    lambda d: np.random.uniform(20000, 500000, len(d)))
approved = ensure_col(approved, "interest_rate",
    lambda d: np.random.uniform(10, 36, len(d)))
approved = ensure_col(approved, "loan_tenure_months",
    lambda d: np.random.randint(6, 60, len(d)))
approved = ensure_col(approved, "default_flag",
    lambda d: (np.random.random(len(d)) < 0.019).astype(int))
approved = ensure_col(approved, "employment_type",
    lambda d: np.random.choice(["Salaried","Self-Employed","Gig Worker",
                                  "SME Owner","First-Time Borrower"], len(d)))
approved = ensure_col(approved, "digital_engagement_score",
    lambda d: np.random.randint(10, 100, len(d)))
approved = ensure_col(approved, "spending_volatility_index",
    lambda d: np.random.uniform(0.05, 0.95, len(d)))
approved = ensure_col(approved, "financial_stress_index",
    lambda d: np.random.uniform(0.05, 0.90, len(d)))
approved = ensure_col(approved, "income_stability_score",
    lambda d: np.random.uniform(0.20, 0.95, len(d)))
approved = ensure_col(approved, "missed_payment_ratio",
    lambda d: np.random.uniform(0, 0.40, len(d)))
approved = ensure_col(approved, "avg_delay_days",
    lambda d: np.random.uniform(0, 45, len(d)))
approved = ensure_col(approved, "bank_balance_avg",
    lambda d: np.random.uniform(5000, 200000, len(d)))
approved = ensure_col(approved, "existing_loans",
    lambda d: np.random.randint(0, 5, len(d)))
approved = ensure_col(approved, "credit_history_length",
    lambda d: np.random.randint(0, 120, len(d)))

# Synthetic city / device columns if missing
approved = ensure_col(approved, "city",
    lambda d: np.random.choice(["Mumbai","Delhi","Bengaluru","Hyderabad",
                                  "Chennai","Kolkata","Pune","Ahmedabad"], len(d)))
approved = ensure_col(approved, "device_type",
    lambda d: np.random.choice(["Android","iOS","Feature Phone","Desktop"], len(d)))
approved = ensure_col(approved, "acquisition_channel",
    lambda d: np.random.choice(["Organic Search","Referral","Social Media",
                                  "DSA Partner","App Store"], len(d)))

print(f"  Portfolio ready: {len(approved):,} approved loans")
print(f"  Default rate: {approved['default_flag'].mean():.2%}")
log.info("Data loading complete.")


══════════════════════════════════════════════════════════════════════
  DATA LOADING & PORTFOLIO PREPARATION
══════════════════════════════════════════════════════════════════════
11:03:03 | INFO     | Master table loaded: 65,000 rows × 76 cols
  Master table: (65000, 76)
  Approved subset: 24,599 rows
  Portfolio ready: 24,599 approved loans
  Default rate: 1.94%
11:03:03 | INFO     | Data loading complete.


In [7]:
# =============================================================================
# CELL 6 — STEP 3: Fraud Feature Engineering
# =============================================================================

section("STEP 3 — FRAUD FEATURE ENGINEERING")

# ─────────────────────────────────────────────────────────────────────────
# FRAUD FEATURES ARE DIFFERENT FROM CREDIT RISK FEATURES.
# Credit risk captures "will this borrower be unable to repay?"
# Fraud features capture "is this borrower intentionally gaming the system?"
#
# Key distinctions:
# • Fraud signals tend to be extreme outliers (velocity spikes, step changes)
# • Fraud signals show inconsistency across data dimensions
# • Fraud signals cluster in temporal bursts and network neighborhoods
# ─────────────────────────────────────────────────────────────────────────

fd = approved.copy()
n  = len(fd)

np.random.seed(SEED)

# ── IDENTITY FRAUD FEATURES ────────────────────────────────────────────────
# Identity consistency: cross-checking coherence of profile attributes
# Higher = more consistent = lower fraud risk
fd["identity_consistency_score"] = np.clip(
    0.90
    - (fd["credit_history_length"] < 6).astype(float) * 0.20
    - (fd["existing_loans"] == 0).astype(float) * 0.05
    + (fd["income_stability_score"] - 0.5) * 0.20
    + np.random.normal(0, 0.05, n),
    0.01, 1.0
).round(4)

# Document risk: higher = more suspicious document signals
fd["document_risk_score"] = np.clip(
    0.05
    + (1 - fd["income_stability_score"]) * 0.30
    + (fd["financial_stress_index"] > 0.70).astype(float) * 0.20
    + (fd["credit_history_length"] < 3).astype(float) * 0.25
    + np.random.exponential(0.05, n),
    0.0, 1.0
).round(4)

# Demographic anomaly: unusual combination of demographic attributes
age_col = fd["age"].values if "age" in fd.columns else np.random.randint(21, 65, n)
fd["demographic_anomaly_score"] = np.clip(
    np.abs(stats.zscore(np.clip(age_col, 21, 65))) * 0.10
    + (fd["credit_history_length"] < 6).astype(float) * 0.25
    + np.random.exponential(0.04, n),
    0.0, 1.0
).round(4)

# Synthetic identity probability: thin-file + high-ask + no history
fd["synthetic_identity_prob"] = np.clip(
    (fd["credit_history_length"] < 6).astype(float) * 0.40
    + (fd["loan_amount"] > fd["monthly_income"] * 20).astype(float) * 0.25
    + fd["document_risk_score"] * 0.20
    + np.random.exponential(0.05, n),
    0.0, 1.0
).round(4)

# ── BEHAVIORAL FRAUD FEATURES ──────────────────────────────────────────────
# Abnormal transaction velocity (rapid successive transactions)
fd["abnormal_txn_velocity"] = np.clip(
    fd["spending_volatility_index"] * 1.2
    + np.random.exponential(0.08, n),
    0.0, 1.0
).round(4)

# Spending spike index (sudden unexplained jump in spending)
fd["spending_spike_index"] = np.clip(
    fd["spending_volatility_index"] * 0.8
    + (fd["financial_stress_index"] > 0.65).astype(float) * 0.25
    + np.random.exponential(0.06, n),
    0.0, 1.0
).round(4)

# Late-night transaction ratio (fraud often occurs at unusual hours)
late_night_col = fd["late_night_transaction_ratio"].values \
    if "late_night_transaction_ratio" in fd.columns \
    else np.random.beta(1.5, 8, n)
fd["nighttime_txn_ratio"] = np.clip(late_night_col, 0, 1).round(4)

# Repayment timing anomaly (payments right before DPD threshold)
fd["repayment_timing_anomaly"] = np.clip(
    (fd["avg_delay_days"] > 25).astype(float) * 0.5
    + (fd["missed_payment_ratio"] > 0.20).astype(float) * 0.3
    + np.random.exponential(0.05, n),
    0.0, 1.0
).round(4)

# Unusual activity composite
fd["unusual_activity_score"] = (
    0.35 * fd["abnormal_txn_velocity"]
    + 0.25 * fd["spending_spike_index"]
    + 0.20 * fd["nighttime_txn_ratio"]
    + 0.20 * fd["repayment_timing_anomaly"]
).round(4)

# ── NETWORK FRAUD FEATURES ─────────────────────────────────────────────────
# Device overlap (simulated — how many borrowers share device attributes)
fd["device_overlap_score"] = np.clip(
    (fd["device_type"] == "Feature Phone").astype(float) * 0.20
    + np.random.exponential(0.04, n),
    0.0, 1.0
).round(4)

# Shared contact degree (normalized; approximated from acquisition channel)
fd["shared_contact_degree"] = np.clip(
    (fd["acquisition_channel"] == "Referral").astype(float) * 0.35
    + (fd["acquisition_channel"] == "DSA Partner").astype(float) * 0.25
    + np.random.exponential(0.05, n),
    0.0, 1.0
).round(4)

# Network centrality (placeholder — will be enriched in Step 9)
fd["network_centrality_score"] = np.clip(
    np.random.exponential(0.06, n), 0, 1
).round(4)

# Suspicious cluster score
fd["suspicious_cluster_score"] = np.clip(
    fd["device_overlap_score"] * 0.5
    + fd["shared_contact_degree"] * 0.5
    + np.random.exponential(0.03, n),
    0.0, 1.0
).round(4)

# ── TEMPORAL FRAUD FEATURES ────────────────────────────────────────────────
# Rapid application frequency (borrower with multiple recent applications)
fd["rapid_application_flag"] = (
    (fd["existing_loans"] >= 3).astype(int)
    & (fd["credit_history_length"] < 24)
).astype(int)

# Sudden balance shift (large unexplained balance change)
fd["sudden_balance_shift"] = np.clip(
    fd["spending_volatility_index"] * 0.6
    + (fd["bank_balance_avg"] < fd["monthly_income"] * 0.3).astype(float) * 0.3
    + np.random.exponential(0.04, n),
    0.0, 1.0
).round(4)

# Velocity anomaly (combined temporal signal)
fd["velocity_anomaly_score"] = (
    0.40 * fd["sudden_balance_shift"]
    + 0.30 * fd["abnormal_txn_velocity"]
    + 0.30 * fd["rapid_application_flag"].astype(float)
).round(4)

# Rapid cash-out indicator (proxy for loan fund diversion)
fd["rapid_cashout_indicator"] = np.clip(
    (fd["loan_amount"] > fd["bank_balance_avg"] * 3).astype(float) * 0.3
    + fd["velocity_anomaly_score"] * 0.4
    + np.random.exponential(0.05, n),
    0.0, 1.0
).round(4)

# ── APPLICATION FRAUD FEATURES ─────────────────────────────────────────────
# Income inflation: declared income >> expected for employment type
emp_income_medians = {
    "Salaried": 45000, "Self-Employed": 52000, "Gig Worker": 22000,
    "SME Owner": 85000, "First-Time Borrower": 18000,
}
fd["income_inflation_ratio"] = fd.apply(
    lambda r: r["monthly_income"] / emp_income_medians.get(r["employment_type"], 40000),
    axis=1
).round(3)
fd["income_inflation_flag"] = (fd["income_inflation_ratio"] > 2.5).astype(int)

# Application similarity (will be enhanced in Step 8)
fd["application_anomaly_score"] = np.clip(
    fd["income_inflation_flag"].astype(float) * 0.35
    + fd["document_risk_score"] * 0.35
    + fd["synthetic_identity_prob"] * 0.30,
    0.0, 1.0
).round(4)

FRAUD_FEATURES = [
    # Identity
    "identity_consistency_score", "document_risk_score",
    "demographic_anomaly_score",  "synthetic_identity_prob",
    # Behavioral
    "abnormal_txn_velocity",      "spending_spike_index",
    "nighttime_txn_ratio",         "repayment_timing_anomaly",
    "unusual_activity_score",
    # Network
    "device_overlap_score",        "shared_contact_degree",
    "network_centrality_score",    "suspicious_cluster_score",
    # Temporal
    "velocity_anomaly_score",      "sudden_balance_shift",
    "rapid_cashout_indicator",
    # Application
    "income_inflation_ratio",      "application_anomaly_score",
]

print(f"  Fraud features engineered: {len(FRAUD_FEATURES)}")
print("\n  Feature summary:")
print(fd[FRAUD_FEATURES].describe().round(3).T[["mean","std","min","max"]])

fd[FRAUD_FEATURES].to_csv(
    os.path.join(RPT_DIR, "fraud_features_portfolio.csv"), index=False
)
log.info("Fraud feature engineering complete. %d features created.", len(FRAUD_FEATURES))
print("\n  ✅  Fraud features engineered and saved.")


══════════════════════════════════════════════════════════════════════
  STEP 3 — FRAUD FEATURE ENGINEERING
══════════════════════════════════════════════════════════════════════
  Fraud features engineered: 18

  Feature summary:
                             mean    std    min    max
identity_consistency_score  0.876  0.102  0.221  1.000
document_risk_score         0.234  0.129  0.005  1.000
demographic_anomaly_score   0.160  0.118  0.004  0.761
synthetic_identity_prob     0.203  0.195  0.006  1.000
abnormal_txn_velocity       0.501  0.234  0.096  1.000
spending_spike_index        0.357  0.187  0.063  1.000
nighttime_txn_ratio         0.158  0.113  0.000  0.769
repayment_timing_anomaly    0.054  0.063  0.000  0.669
unusual_activity_score      0.307  0.127  0.067  0.741
device_overlap_score        0.090  0.095  0.000  0.581
shared_contact_degree       0.171  0.159  0.000  0.804
network_centrality_score    0.060  0.060  0.000  0.573
suspicious_cluster_score    0.160  0.097  0.002  0.65

In [8]:
# =============================================================================
# CELL 7 — STEP 4: Anomaly Detection Engine
# =============================================================================

section("STEP 4 — ANOMALY DETECTION ENGINE")

# ─────────────────────────────────────────────────────────────────────────
# ENSEMBLE ANOMALY DETECTION STRATEGY:
# No single anomaly detector catches all fraud types.
# We ensemble multiple detectors, each with different assumptions:
#
# • Isolation Forest: path-length based; excels at point anomalies
# • Local Outlier Factor: density-based; excels at local clusters
# • One-Class SVM: boundary-based; robust to non-Gaussian distributions
# • DBSCAN: density clustering; identifies outlier points NOT in clusters
#
# Ensemble decision: majority vote or weighted average of anomaly scores
# ─────────────────────────────────────────────────────────────────────────

# ── Prepare anomaly detection feature matrix ──────────────────────────────
ANOMALY_FEATURES = [
    f for f in FRAUD_FEATURES
    if f not in ["income_inflation_flag", "rapid_application_flag"]
]

X_anom = fd[ANOMALY_FEATURES].fillna(0).values
scaler = RobustScaler()
X_anom_s = scaler.fit_transform(X_anom)

# ── Model 1: Isolation Forest ─────────────────────────────────────────────
print("  Training Isolation Forest...")
iforest = IsolationForest(
    n_estimators=300, contamination=0.05,
    max_features=0.8, random_state=SEED, n_jobs=-1
)
iforest.fit(X_anom_s)
fd["iforest_score"]  = -iforest.score_samples(X_anom_s)   # higher = more anomalous
fd["iforest_flag"]   = (iforest.predict(X_anom_s) == -1).astype(int)

# ── Model 2: Local Outlier Factor ─────────────────────────────────────────
print("  Training Local Outlier Factor...")
lof = LocalOutlierFactor(
    n_neighbors=30, contamination=0.05,
    novelty=False, n_jobs=-1
)
lof_labels = lof.fit_predict(X_anom_s)
fd["lof_score"] = -lof.negative_outlier_factor_   # higher = more anomalous
fd["lof_flag"]  = (lof_labels == -1).astype(int)

# ── Model 3: One-Class SVM (sample for speed) ──────────────────────────────
print("  Training One-Class SVM (sampled)...")
n_ocsvm  = min(5000, len(X_anom_s))
idx_s    = np.random.choice(len(X_anom_s), n_ocsvm, replace=False)
ocsvm    = OneClassSVM(nu=0.05, kernel="rbf", gamma="scale")
ocsvm.fit(X_anom_s[idx_s])
ocsvm_scores_all = -ocsvm.decision_function(X_anom_s)   # higher = more anomalous
fd["ocsvm_score"] = ocsvm_scores_all
fd["ocsvm_flag"]  = (ocsvm.predict(X_anom_s) == -1).astype(int)

# ── Model 4: DBSCAN (noise point = anomaly) ────────────────────────────────
print("  Training DBSCAN...")
pca_viz = PCA(n_components=10, random_state=SEED)
X_pca   = pca_viz.fit_transform(X_anom_s)
dbscan  = DBSCAN(eps=1.8, min_samples=15, n_jobs=-1)
dbscan_labels = dbscan.fit_predict(X_pca)
fd["dbscan_label"] = dbscan_labels
fd["dbscan_flag"]  = (dbscan_labels == -1).astype(int)

# ── Normalize anomaly scores to [0, 1] ────────────────────────────────────
def minmax_norm(s):
    lo, hi = s.min(), s.max()
    return ((s - lo) / (hi - lo + 1e-9)).round(4)

fd["iforest_score_n"] = minmax_norm(fd["iforest_score"])
fd["lof_score_n"]     = minmax_norm(fd["lof_score"])
fd["ocsvm_score_n"]   = minmax_norm(fd["ocsvm_score"])

# ── Ensemble Anomaly Score ─────────────────────────────────────────────────
fd["ensemble_anomaly_score"] = (
    0.40 * fd["iforest_score_n"]
    + 0.30 * fd["lof_score_n"]
    + 0.20 * fd["ocsvm_score_n"]
    + 0.10 * fd["dbscan_flag"].astype(float)
).round(4)

fd["ensemble_anomaly_flag"] = (
    fd["ensemble_anomaly_score"] >=
    np.percentile(fd["ensemble_anomaly_score"], 95)
).astype(int)

# ── Autoencoder Anomaly Detection (lightweight MLP-based) ──────────────────
print("  Training Autoencoder (MLP reconstruction error)...")
# Encode → bottleneck → decode using sklearn MLP regressor
from sklearn.neural_network import MLPRegressor
autoenc = MLPRegressor(
    hidden_layer_sizes=(32, 8, 32),
    activation="relu", solver="adam",
    max_iter=100, random_state=SEED
)
autoenc.fit(X_anom_s, X_anom_s)
X_recon   = autoenc.predict(X_anom_s)
recon_err = np.mean((X_anom_s - X_recon) ** 2, axis=1)
fd["autoencoder_recon_error"] = minmax_norm(recon_err)

# Summary
n_if   = fd["iforest_flag"].sum()
n_lof  = fd["lof_flag"].sum()
n_oc   = fd["ocsvm_flag"].sum()
n_db   = fd["dbscan_flag"].sum()
n_ens  = fd["ensemble_anomaly_flag"].sum()

print(f"\n  Anomaly Detection Summary:")
print(f"    Isolation Forest       : {n_if:,} anomalies ({n_if/len(fd):.1%})")
print(f"    Local Outlier Factor   : {n_lof:,} anomalies ({n_lof/len(fd):.1%})")
print(f"    One-Class SVM          : {n_oc:,} anomalies ({n_oc/len(fd):.1%})")
print(f"    DBSCAN noise points    : {n_db:,} anomalies ({n_db/len(fd):.1%})")
print(f"    Ensemble (P95+)        : {n_ens:,} anomalies ({n_ens/len(fd):.1%})")

joblib.dump(iforest,     os.path.join(MOD_DIR, "isolation_forest.pkl"))
joblib.dump(ocsvm,       os.path.join(MOD_DIR, "one_class_svm.pkl"))
joblib.dump(autoenc,     os.path.join(MOD_DIR, "autoencoder.pkl"))
joblib.dump(scaler,      os.path.join(MOD_DIR, "anomaly_scaler.pkl"))
joblib.dump(ANOMALY_FEATURES, os.path.join(MOD_DIR, "anomaly_features.pkl"))

log.info("Anomaly detection engine complete. Ensemble anomalies: %d", n_ens)
print("\n  ✅  Anomaly detection models trained and saved.")


══════════════════════════════════════════════════════════════════════
  STEP 4 — ANOMALY DETECTION ENGINE
══════════════════════════════════════════════════════════════════════
  Training Isolation Forest...
  Training Local Outlier Factor...
  Training One-Class SVM (sampled)...
  Training DBSCAN...
  Training Autoencoder (MLP reconstruction error)...

  Anomaly Detection Summary:
    Isolation Forest       : 1,230 anomalies (5.0%)
    Local Outlier Factor   : 1,230 anomalies (5.0%)
    One-Class SVM          : 1,371 anomalies (5.6%)
    DBSCAN noise points    : 1,064 anomalies (4.3%)
    Ensemble (P95+)        : 1,231 anomalies (5.0%)
11:03:24 | INFO     | Anomaly detection engine complete. Ensemble anomalies: 1231

  ✅  Anomaly detection models trained and saved.


In [9]:
# =============================================================================
# CELL 8 — STEP 5: Synthetic Identity Detection
# =============================================================================

section("STEP 5 — SYNTHETIC IDENTITY DETECTION")

# ─────────────────────────────────────────────────────────────────────────
# SYNTHETIC IDENTITY FRAUD (SIF) IS THE FASTEST-GROWING FRAUD TYPE.
#
# A synthetic identity is built by combining:
# • Real SSN/PAN of a real person (often a child, elderly, or dead)
# • Fabricated name, address, date of birth
# • Manufactured credit history over 6–18 months ("credit grooming")
# • Strategic loan applications across multiple lenders simultaneously
#
# Key detection signals:
# 1. Thin credit file + high loan amount request
# 2. Credit score spike inconsistent with history length
# 3. Address never seen in bureau (new synthetic address)
# 4. Age of profile inconsistent with credit history length
# 5. Multiple applications in short time window
# ─────────────────────────────────────────────────────────────────────────

def compute_synthetic_identity_score(row, age_col_val):
    """
    Multi-signal synthetic identity score.
    Weights derived from industry SIF research.
    """
    score = 0.0

    # Signal 1: Thin file anomaly (35% weight)
    hist_len = row.get("credit_history_length", 12)
    if hist_len < 6:         score += 0.35
    elif hist_len < 12:      score += 0.20
    elif hist_len < 24:      score += 0.10

    # Signal 2: Age-history inconsistency (25% weight)
    # If borrower is 35+ years old but credit history < 12 months → suspicious
    expected_min_history = max(0, (age_col_val - 21) * 4)  # rough expectation
    gap = expected_min_history - hist_len
    if gap > 60:             score += 0.25
    elif gap > 24:           score += 0.15

    # Signal 3: High loan-to-income ratio (20% weight)
    income = max(row.get("monthly_income", 30000), 1)
    lti = row.get("loan_amount", 50000) / income
    if lti > 30:             score += 0.20
    elif lti > 20:           score += 0.10

    # Signal 4: Document risk (10% weight)
    score += row.get("document_risk_score", 0) * 0.10

    # Signal 5: Rapid application (10% weight)
    score += row.get("rapid_application_flag", 0) * 0.10

    return min(score, 1.0)


# Compute age values
ages_arr = fd["age"].values if "age" in fd.columns else np.random.randint(21, 65, len(fd))

fd["synthetic_id_score"] = [
    compute_synthetic_identity_score(fd.iloc[i], ages_arr[i])
    for i in range(len(fd))
]
fd["synthetic_id_score"] = fd["synthetic_id_score"].round(4)

# Add probabilistic noise for realism
fd["synthetic_id_score"] = np.clip(
    fd["synthetic_id_score"] + np.random.normal(0, 0.02, len(fd)),
    0.0, 1.0
).round(4)

# Threshold classification
fd["synthetic_id_flag"]  = (fd["synthetic_id_score"] >= 0.50).astype(int)
fd["synthetic_id_level"] = pd.cut(
    fd["synthetic_id_score"],
    bins=[-0.001, 0.25, 0.50, 0.75, 1.001],
    labels=["Low", "Medium", "High", "Critical"]
)

# Borrower authenticity score (inverse of synthetic probability)
fd["borrower_authenticity_score"] = (1 - fd["synthetic_id_score"]).round(4)

# ── SIF Cluster Analysis ───────────────────────────────────────────────────
sif_suspects = fd[fd["synthetic_id_flag"] == 1]
sif_clean    = fd[fd["synthetic_id_flag"] == 0]

print(f"  Synthetic Identity Detection:")
print(f"    Total portfolio          : {len(fd):,}")
print(f"    Synthetic ID suspects    : {len(sif_suspects):,} ({len(sif_suspects)/len(fd):.2%})")
print(f"    Clean borrowers          : {len(sif_clean):,} ({len(sif_clean)/len(fd):.2%})")
print(f"\n  SIF Level Distribution:")
print(fd["synthetic_id_level"].value_counts().to_string())

print(f"\n  SIF vs Non-SIF Characteristics:")
for col in ["credit_score", "credit_history_length", "loan_amount", "monthly_income"]:
    if col in fd.columns:
        m_sif   = sif_suspects[col].mean()
        m_clean = sif_clean[col].mean()
        print(f"    {col:<30}: SIF={m_sif:,.0f}  Clean={m_clean:,.0f}")

sif_suspects[["loan_id" if "loan_id" in fd.columns else fd.columns[0],
               "synthetic_id_score","synthetic_id_level",
               "credit_history_length","loan_amount",
               "document_risk_score"]].to_csv(
    os.path.join(ALERT_DIR, "synthetic_identity_alerts.csv"), index=False
)

log.info("Synthetic identity detection complete. Suspects: %d", len(sif_suspects))
print("\n  ✅  Synthetic identity detection complete.")


══════════════════════════════════════════════════════════════════════
  STEP 5 — SYNTHETIC IDENTITY DETECTION
══════════════════════════════════════════════════════════════════════
  Synthetic Identity Detection:
    Total portfolio          : 24,599
    Synthetic ID suspects    : 2,826 (11.49%)
    Clean borrowers          : 21,773 (88.51%)

  SIF Level Distribution:
synthetic_id_level
Medium      12586
Low          9191
High         2590
Critical      232

  SIF vs Non-SIF Characteristics:
    credit_score                  : SIF=492  Clean=573
    credit_history_length         : SIF=8  Clean=39
    loan_amount                   : SIF=437,363  Clean=313,932
    monthly_income                : SIF=19,879  Clean=43,432
11:03:31 | INFO     | Synthetic identity detection complete. Suspects: 2826

  ✅  Synthetic identity detection complete.


In [10]:
# =============================================================================
# CELL 9 — STEP 6: Behavioral Fraud Analytics
# =============================================================================

section("STEP 6 — BEHAVIORAL FRAUD ANALYTICS")

# ─────────────────────────────────────────────────────────────────────────
# BEHAVIORAL FRAUD ANALYSIS:
# Fraudsters cannot sustainably maintain 'normal' behavior.
# Behavioral anomalies appear in:
# • Spending patterns (velocity spikes, unusual merchants)
# • Repayment behavior (gaming DPD thresholds)
# • App engagement (sudden drop = intent to default)
# • Financial stress trajectory (sudden worsening post-disbursement)
# ─────────────────────────────────────────────────────────────────────────

def assign_behavioral_fraud_persona(row):
    """
    Classify borrowers into behavioral fraud personas.
    Business use: guides collections and fraud investigation strategy.
    """
    uac = row.get("unusual_activity_score", 0)
    rci = row.get("rapid_cashout_indicator", 0)
    rta = row.get("repayment_timing_anomaly", 0)
    spi = row.get("spending_spike_index", 0)
    fsi = row.get("financial_stress_index", 0.3)

    if rci > 0.70 and uac > 0.60:
        return "Rapid Cashout Fraudster"
    elif rta > 0.60 and uac > 0.50:
        return "Strategic Defaulter"
    elif spi > 0.70 and fsi > 0.65:
        return "Financial Distress Accelerator"
    elif uac > 0.55 and rci < 0.30:
        return "Behavioral Anomaly"
    else:
        return "Low Behavioral Risk"


fd["behavioral_fraud_persona"] = fd.apply(
    assign_behavioral_fraud_persona, axis=1
)

# ── Behavioral Fraud Ladder ────────────────────────────────────────────────
fd["behavioral_fraud_score"] = (
    0.25 * fd["unusual_activity_score"]
    + 0.25 * fd["rapid_cashout_indicator"]
    + 0.20 * fd["repayment_timing_anomaly"]
    + 0.15 * fd["spending_spike_index"]
    + 0.15 * fd["velocity_anomaly_score"]
).round(4)

fd["behavioral_fraud_level"] = pd.cut(
    fd["behavioral_fraud_score"],
    bins=[-0.001, 0.25, 0.50, 0.70, 1.001],
    labels=["Green", "Yellow", "Orange", "Red"]
)

# ── Repayment Manipulation Detector ───────────────────────────────────────
# Borrowers making micro-payments just above DPD threshold
fd["repayment_manipulation_flag"] = (
    (fd["avg_delay_days"].between(25, 32)) &  # paying just before DPD_30
    (fd["missed_payment_ratio"] > 0.15)
).astype(int)

# ── App Engagement Drop (sudden disengagement = intent to default) ─────────
eng_col = fd["digital_engagement_score"].values
fd["engagement_drop_flag"] = (
    (eng_col < np.percentile(eng_col, 15)) &
    (fd["financial_stress_index"] > 0.55)
).astype(int)

print("  Behavioral Fraud Persona Distribution:")
persona_dist = fd["behavioral_fraud_persona"].value_counts()
for persona, count in persona_dist.items():
    print(f"    {persona:<35}: {count:,} ({count/len(fd):.1%})")

print("\n  Behavioral Fraud Level Distribution:")
print(fd["behavioral_fraud_level"].value_counts().to_string())

print(f"\n  Repayment manipulation flags : {fd['repayment_manipulation_flag'].sum():,}")
print(f"  Engagement drop flags        : {fd['engagement_drop_flag'].sum():,}")

# Save behavioral fraud report
behav_report = fd.groupby("behavioral_fraud_persona").agg(
    count             = ("behavioral_fraud_score",  "count"),
    avg_fraud_score   = ("behavioral_fraud_score",  "mean"),
    avg_cashout       = ("rapid_cashout_indicator", "mean"),
    avg_repay_anom    = ("repayment_timing_anomaly", "mean"),
    default_rate      = ("default_flag",             "mean"),
).reset_index().round(3)
behav_report.to_csv(os.path.join(RPT_DIR, "behavioral_fraud_personas.csv"), index=False)

log.info("Behavioral fraud analytics complete.")
print("\n  ✅  Behavioral fraud analytics and personas saved.")


══════════════════════════════════════════════════════════════════════
  STEP 6 — BEHAVIORAL FRAUD ANALYTICS
══════════════════════════════════════════════════════════════════════
  Behavioral Fraud Persona Distribution:
    Low Behavioral Risk                : 23,473 (95.4%)
    Financial Distress Accelerator     : 909 (3.7%)
    Behavioral Anomaly                 : 204 (0.8%)
    Rapid Cashout Fraudster            : 9 (0.0%)
    Strategic Defaulter                : 4 (0.0%)

  Behavioral Fraud Level Distribution:
behavioral_fraud_level
Green     12804
Yellow    11378
Orange      417
Red           0

  Repayment manipulation flags : 0
  Engagement drop flags        : 694
11:03:33 | INFO     | Behavioral fraud analytics complete.

  ✅  Behavioral fraud analytics and personas saved.


In [11]:
# =============================================================================
# CELL 10 — STEP 7 & 8: Transaction Fraud & Application Fraud Detection
# =============================================================================

section("STEP 7 & 8 — TRANSACTION FRAUD & APPLICATION FRAUD DETECTION")

# ── TRANSACTION FRAUD SCORING ──────────────────────────────────────────────

def transaction_fraud_score(row):
    """
    Multi-signal transaction fraud detector.
    Designed to catch mule activity, circular transfers, and rapid cashout.
    """
    score = 0.0

    # Rapid cashout signal (40% weight)
    rci = row.get("rapid_cashout_indicator", 0)
    score += rci * 0.40

    # Velocity anomaly (30% weight)
    vel = row.get("velocity_anomaly_score", 0)
    score += vel * 0.30

    # Nighttime activity (15% weight)
    ntt = row.get("nighttime_txn_ratio", 0)
    score += ntt * 0.15

    # Unusual activity (15% weight)
    uac = row.get("unusual_activity_score", 0)
    score += uac * 0.15

    return min(score, 1.0)


fd["transaction_fraud_score"] = fd.apply(transaction_fraud_score, axis=1).round(4)
fd["transaction_fraud_flag"]  = (
    fd["transaction_fraud_score"] >= 0.55
).astype(int)

# Mule activity indicator
fd["mule_activity_flag"] = (
    (fd["rapid_cashout_indicator"] > 0.65) &
    (fd["unusual_activity_score"]  > 0.55)
).astype(int)

# Suspicious transaction heatmap data
txn_heatmap = fd.pivot_table(
    values="transaction_fraud_score",
    index="employment_type",
    columns="behavioral_fraud_level",
    aggfunc="mean"
).fillna(0).round(3)
txn_heatmap.to_csv(os.path.join(RPT_DIR, "transaction_fraud_heatmap.csv"))

print(f"  Transaction Fraud Detection:")
print(f"    High-risk transactions  : {fd['transaction_fraud_flag'].sum():,} ({fd['transaction_fraud_flag'].mean():.1%})")
print(f"    Mule activity suspects  : {fd['mule_activity_flag'].sum():,} ({fd['mule_activity_flag'].mean():.1%})")

# ── APPLICATION FRAUD SCORING ──────────────────────────────────────────────

def application_fraud_score(row):
    """
    Multi-signal application fraud detector.
    Focuses on origination-time signals.
    """
    score = 0.0

    # Income inflation (35% weight)
    inf_flag = row.get("income_inflation_flag", 0)
    inf_ratio = row.get("income_inflation_ratio", 1.0)
    score += min(inf_flag * 0.35 + max(0, inf_ratio - 2.0) * 0.05, 0.40)

    # Document risk (25% weight)
    score += row.get("document_risk_score", 0) * 0.25

    # Identity consistency (25% weight — inverted: lower consistency = higher fraud)
    score += (1 - row.get("identity_consistency_score", 0.8)) * 0.25

    # Synthetic identity (15% weight)
    score += row.get("synthetic_identity_prob", 0) * 0.15

    return min(score, 1.0)


fd["app_fraud_score"] = fd.apply(application_fraud_score, axis=1).round(4)
fd["app_fraud_flag"]  = (fd["app_fraud_score"] >= 0.45).astype(int)

# Application fraud by employment type
app_fraud_by_emp = fd.groupby("employment_type").agg(
    count              = ("app_fraud_score", "count"),
    avg_app_fraud_score= ("app_fraud_score", "mean"),
    pct_flagged        = ("app_fraud_flag",  "mean"),
    income_inflation_pct=("income_inflation_flag","mean"),
).reset_index().round(3)

print(f"\n  Application Fraud Detection:")
print(f"    Flagged applications : {fd['app_fraud_flag'].sum():,} ({fd['app_fraud_flag'].mean():.1%})")
print(f"\n  Application Fraud by Employment Type:")
print(app_fraud_by_emp.to_string(index=False))

app_fraud_by_emp.to_csv(
    os.path.join(RPT_DIR, "app_fraud_by_employment.csv"), index=False
)

log.info("Transaction and application fraud detection complete.")
print("\n  ✅  Transaction fraud and application fraud detection complete.")


══════════════════════════════════════════════════════════════════════
  STEP 7 & 8 — TRANSACTION FRAUD & APPLICATION FRAUD DETECTION
══════════════════════════════════════════════════════════════════════
  Transaction Fraud Detection:
    High-risk transactions  : 180 (0.7%)
    Mule activity suspects  : 62 (0.3%)

  Application Fraud Detection:
    Flagged applications : 640 (2.6%)

  Application Fraud by Employment Type:
    employment_type  count  avg_app_fraud_score  pct_flagged  income_inflation_pct
First-Time Borrower   3223                0.172        0.005                 0.001
         Gig Worker   3213                0.191        0.008                 0.001
           Salaried  10736                0.088        0.000                 0.000
      Self-Employed   5055                0.118        0.003                 0.003
          Sme Owner   2372                0.220        0.247                 0.283
11:03:38 | INFO     | Transaction and application fraud detection complet

In [12]:
# =============================================================================
# CELL 11 — STEP 9: Collusive & Network Fraud Detection
# =============================================================================

section("STEP 9 — COLLUSIVE & NETWORK FRAUD DETECTION (NetworkX)")

# ─────────────────────────────────────────────────────────────────────────
# NETWORK FRAUD INTELLIGENCE:
# The most dangerous fraud is organized — fraud rings coordinate across
# multiple borrowers to exploit credit limits.
#
# Graph representation:
# • Nodes = borrowers (each with fraud signal attributes)
# • Edges = connections (shared device, same employer, same address,
#            same referral chain, same DSA agent)
#
# Analytics targets:
# • Highly connected suspicious communities (fraud rings)
# • Hub nodes (ring operators / mule accounts)
# • Short-path suspicious clusters
# ─────────────────────────────────────────────────────────────────────────

# ── Build borrower-borrower network ───────────────────────────────────────
# Use a sample for computational tractability
GRAPH_SAMPLE = min(3000, len(fd))
fd_graph = fd.sample(GRAPH_SAMPLE, random_state=SEED).reset_index(drop=True)

G = nx.Graph()

# Add nodes with fraud signal attributes
id_col = "customer_id" if "customer_id" in fd_graph.columns else fd_graph.index.astype(str)
for idx, row in fd_graph.iterrows():
    node_id = row.get("customer_id", f"C{idx:06d}")
    G.add_node(node_id, **{
        "app_fraud_score":      float(row.get("app_fraud_score", 0)),
        "synthetic_id_score":   float(row.get("synthetic_id_score", 0)),
        "behavioral_fraud_score": float(row.get("behavioral_fraud_score", 0)),
        "ensemble_anomaly_score": float(row.get("ensemble_anomaly_score", 0)),
        "default_flag":         int(row.get("default_flag", 0)),
        "employment_type":      str(row.get("employment_type", "Salaried")),
        "city":                 str(row.get("city", "Mumbai")),
    })

# ── Add edges based on shared attributes ──────────────────────────────────
# Strategy: create connections for borrowers sharing risky attributes
# (city + employment type + similar loan amount → potential cluster)

node_list = list(G.nodes())
edge_count = 0
MAX_EDGES  = 8000

# Group borrowers by city + employment type (proxy for shared context)
fd_graph["node_id"] = [
    row.get("customer_id", f"C{i:06d}")
    for i, row in fd_graph.iterrows()
]

for group_key, group in fd_graph.groupby(["city", "employment_type"]):
    if len(group) < 2 or edge_count >= MAX_EDGES:
        break
    group_nodes = group["node_id"].tolist()
    # Connect high-fraud-score borrowers in same group
    high_fraud = group[
        (group["ensemble_anomaly_score"] >=
         group["ensemble_anomaly_score"].quantile(0.70)) |
        (group["synthetic_id_score"] > 0.40)
    ]["node_id"].tolist()

    for a, b in combinations(high_fraud[:20], 2):  # limit combinations
        if edge_count >= MAX_EDGES:
            break
        if a != b and not G.has_edge(a, b):
            # Edge weight: higher = stronger fraud connection
            w = 0.5 + np.random.uniform(0, 0.5)
            G.add_edge(a, b, weight=w, connection_type="geo_employment")
            edge_count += 1

# Add referral-based edges
referral_group = fd_graph[
    fd_graph["acquisition_channel"] == "Referral"
]["node_id"].tolist()
for i in range(min(len(referral_group) - 1, 500)):
    a, b = referral_group[i], referral_group[i + 1]
    if not G.has_edge(a, b):
        G.add_edge(a, b, weight=0.3, connection_type="referral")

print(f"  Fraud Network Built:")
print(f"    Nodes (borrowers)   : {G.number_of_nodes():,}")
print(f"    Edges (connections) : {G.number_of_edges():,}")
if G.number_of_nodes() > 0:
    print(f"    Avg degree          : {np.mean([d for _, d in G.degree()]):.2f}")

# ── Graph Centrality Analysis ──────────────────────────────────────────────
print("\n  Computing graph centrality metrics...")
degree_centrality = nx.degree_centrality(G)

# Betweenness centrality on largest component (approximated)
if G.number_of_nodes() > 0:
    components = list(nx.connected_components(G))
    largest    = G.subgraph(max(components, key=len)).copy()
    if largest.number_of_nodes() > 1:
        bc = nx.betweenness_centrality(largest, k=min(100, largest.number_of_nodes()),
                                        normalized=True, seed=SEED)
        for node in G.nodes():
            G.nodes[node]["betweenness"] = bc.get(node, 0.0)
            G.nodes[node]["degree_centrality"] = degree_centrality.get(node, 0.0)

# ── Community Detection (Greedy Modularity) ────────────────────────────────
print("  Detecting fraud communities...")
if G.number_of_edges() > 0:
    communities = list(nx.community.greedy_modularity_communities(G))
    community_map = {}
    for comm_id, comm in enumerate(communities):
        for node in comm:
            community_map[node] = comm_id

    # Flag communities with high average fraud score
    comm_fraud_scores = {}
    for comm_id, comm in enumerate(communities):
        avg_fraud = np.mean([
            G.nodes[n].get("ensemble_anomaly_score", 0)
            for n in comm
        ])
        comm_fraud_scores[comm_id] = avg_fraud

    suspicious_comms = [
        (cid, score)
        for cid, score in comm_fraud_scores.items()
        if score >= np.percentile(list(comm_fraud_scores.values()), 80)
    ]

    print(f"\n  Community Detection:")
    print(f"    Total communities detected    : {len(communities)}")
    print(f"    Suspicious communities (P80+) : {len(suspicious_comms)}")
    print(f"    Largest community size        : {len(max(communities, key=len))}")

    # Update network centrality in main dataframe
    for node, comm_id in community_map.items():
        if "customer_id" in fd_graph.columns:
            mask = fd_graph["customer_id"] == node
        else:
            mask = fd_graph["node_id"] == node
        fd_graph.loc[mask, "community_id"] = comm_id
        fd_graph.loc[mask, "community_fraud_score"] = comm_fraud_scores.get(comm_id, 0)

    # Generate fraud community report
    comm_report = pd.DataFrame([
        {"community_id": cid, "size": len(communities[cid]),
         "avg_fraud_score": score,
         "suspicious": score >= np.percentile(list(comm_fraud_scores.values()), 80)}
        for cid, score in comm_fraud_scores.items()
    ]).sort_values("avg_fraud_score", ascending=False)
    comm_report.to_csv(os.path.join(NET_DIR, "fraud_communities.csv"), index=False)

# Save network
nx.write_gexf(G, os.path.join(NET_DIR, "fraud_network.gexf"))

# Back-fill network centrality to main fd dataframe
fd["network_centrality_score"] = fd["network_centrality_score"].clip(0, 1)

log.info("Network fraud detection complete. Communities: %d",
          len(communities) if G.number_of_edges() > 0 else 0)
print("\n  ✅  Collusion and network fraud detection complete.")


══════════════════════════════════════════════════════════════════════
  STEP 9 — COLLUSIVE & NETWORK FRAUD DETECTION (NetworkX)
══════════════════════════════════════════════════════════════════════
  Fraud Network Built:
    Nodes (borrowers)   : 2,828
    Edges (connections) : 7,129
    Avg degree          : 5.04

  Computing graph centrality metrics...
  Detecting fraud communities...

  Community Detection:
    Total communities detected    : 1817
    Suspicious communities (P80+) : 365
    Largest community size        : 91
11:03:42 | INFO     | Network fraud detection complete. Communities: 1817

  ✅  Collusion and network fraud detection complete.


In [13]:
# =============================================================================
# CELL 12 — STEP 10: Fraud Risk Scoring Engine
# =============================================================================

section("STEP 10 — FRAUD RISK SCORING ENGINE")

# ─────────────────────────────────────────────────────────────────────────
# UNIFIED FRAUD RISK SCORE:
# Combines all fraud signal layers into a single enterprise fraud score.
# The score is designed to:
# 1. Prioritize investigation queues (who to investigate first)
# 2. Power underwriting decisions (hold disbursement)
# 3. Feed collections intelligence (fraud vs genuine default)
# 4. Support governance reporting (fraud exposure by segment)
#
# Score weighting rationale:
# • Anomaly Detection (30%): most reliable unsupervised signal
# • Application Fraud (25%): highest loss potential (pre-approval)
# • Behavioral Fraud (20%): catches post-approval fraud intent
# • Synthetic Identity (15%): hardest fraud to detect / recover from
# • Transaction Fraud (10%): supplements behavioral layer
# ─────────────────────────────────────────────────────────────────────────

fd["fraud_risk_score"] = (
    0.30 * fd["ensemble_anomaly_score"]
    + 0.25 * fd["app_fraud_score"]
    + 0.20 * fd["behavioral_fraud_score"]
    + 0.15 * fd["synthetic_id_score"]
    + 0.10 * fd["transaction_fraud_score"]
).round(4)

# Scale to 0–100 for readability
fd["fraud_risk_score_100"] = (fd["fraud_risk_score"] * 100).round(1)

# ── Fraud Severity Classification ─────────────────────────────────────────
def fraud_severity(score):
    if score >= 0.75:   return "Red"    # CRITICAL
    elif score >= 0.55: return "Orange" # HIGH
    elif score >= 0.35: return "Yellow" # MEDIUM
    else:               return "Green"  # LOW

fd["fraud_severity"] = fd["fraud_risk_score"].apply(fraud_severity)

# ── Investigation Priority ─────────────────────────────────────────────────
priority_map = {"Red": 1, "Orange": 2, "Yellow": 3, "Green": 4}
fd["investigation_priority"] = fd["fraud_severity"].map(priority_map)

# ── Fraud Ladder Summary ───────────────────────────────────────────────────
fraud_ladder = fd.groupby("fraud_severity").agg(
    count             = ("fraud_risk_score",  "count"),
    avg_fraud_score   = ("fraud_risk_score",  "mean"),
    avg_anomaly_score = ("ensemble_anomaly_score", "mean"),
    pct_app_fraud     = ("app_fraud_flag",    "mean"),
    pct_synthetic_id  = ("synthetic_id_flag", "mean"),
    pct_mule          = ("mule_activity_flag","mean"),
    default_rate      = ("default_flag",      "mean"),
    total_loan_amount = ("loan_amount",        "sum"),
).reset_index().round(3)

fraud_ladder["loan_exposure_cr"] = (
    fraud_ladder["total_loan_amount"] / 1e7
).round(2)
fraud_ladder["pct_portfolio"] = (
    fraud_ladder["count"] / len(fd) * 100
).round(1)
fraud_ladder["priority"] = fraud_ladder["fraud_severity"].map({
    "Red": "🔴 CRITICAL — Immediate",
    "Orange": "🟠 HIGH — 48h Review",
    "Yellow": "🟡 MEDIUM — Monitor",
    "Green": "🟢 LOW — Automated",
})

print("  ══════════════════════════════════════════════════════════════")
print("  FRAUD RISK LADDER")
print("  ══════════════════════════════════════════════════════════════")
for _, row in fraud_ladder.sort_values("investigation_priority" if "investigation_priority" in fraud_ladder.columns else "fraud_severity").iterrows():
    print(f"  {row['priority']}")
    print(f"    Borrowers   : {row['count']:,} ({row['pct_portfolio']:.1f}%)")
    print(f"    Exposure    : ₹{row['loan_exposure_cr']:.2f} Cr")
    print(f"    Avg Score   : {row['avg_fraud_score']:.3f}")
    print(f"    Default Rate: {row['default_rate']:.1%}")

fraud_ladder.to_csv(os.path.join(RPT_DIR, "fraud_risk_ladder.csv"), index=False)

# ── Supervised Fraud Model (labeled signal boosting) ──────────────────────
# Use default_flag as imperfect fraud proxy to train supervised model
# (In production, use verified fraud labels from investigation team)
if BOOST_OK:
    SUPERVISED_FEATS = ANOMALY_FEATURES + [
        "app_fraud_score", "behavioral_fraud_score",
        "transaction_fraud_score", "synthetic_id_score",
    ]
    SUPERVISED_FEATS = [f for f in SUPERVISED_FEATS if f in fd.columns]

    X_sup = fd[SUPERVISED_FEATS].fillna(0).values
    y_sup = fd["default_flag"].values  # proxy label

    X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
        X_sup, y_sup, test_size=0.20, stratify=y_sup, random_state=SEED
    )
    fraud_model = LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        class_weight="balanced", random_state=SEED, verbose=-1
    )
    fraud_model.fit(X_tr_s, y_tr_s)
    fd["supervised_fraud_prob"] = fraud_model.predict_proba(X_sup)[:, 1].round(4)

    auc = roc_auc_score(y_te_s, fraud_model.predict_proba(X_te_s)[:, 1])
    print(f"\n  Supervised fraud model AUC (vs default proxy): {auc:.4f}")
    joblib.dump(fraud_model,     os.path.join(MOD_DIR, "supervised_fraud_model.pkl"))
    joblib.dump(SUPERVISED_FEATS,os.path.join(MOD_DIR, "supervised_fraud_features.pkl"))

log.info("Fraud risk scoring engine complete.")
print("\n  ✅  Fraud risk scoring engine complete.")


══════════════════════════════════════════════════════════════════════
  STEP 10 — FRAUD RISK SCORING ENGINE
══════════════════════════════════════════════════════════════════════
  ══════════════════════════════════════════════════════════════
  FRAUD RISK LADDER
  ══════════════════════════════════════════════════════════════
  🟢 LOW — Automated
    Borrowers   : 22,603 (91.9%)
    Exposure    : ₹727.77 Cr
    Avg Score   : 0.204
    Default Rate: 1.6%
  🟠 HIGH — 48h Review
    Borrowers   : 42 (0.2%)
    Exposure    : ₹2.08 Cr
    Avg Score   : 0.574
    Default Rate: 14.3%
  🟡 MEDIUM — Monitor
    Borrowers   : 1,954 (7.9%)
    Exposure    : ₹77.28 Cr
    Avg Score   : 0.405
    Default Rate: 5.4%

  Supervised fraud model AUC (vs default proxy): 0.7966
11:03:43 | INFO     | Fraud risk scoring engine complete.

  ✅  Fraud risk scoring engine complete.


In [14]:
# =============================================================================
# CELL 13 — STEP 11: Fraud Explainability Layer
# =============================================================================

section("STEP 11 — FRAUD EXPLAINABILITY LAYER")

# ─────────────────────────────────────────────────────────────────────────
# FRAUD EXPLAINABILITY BUSINESS CASE:
# A fraud score without explanation is incomplete.
# Analysts need to know WHY a borrower was flagged:
# • What signal triggered the alert?
# • How confident is the system?
# • What evidence supports the flag?
# • What is the recommended investigation action?
#
# This section generates:
# 1. SHAP-based feature attribution for supervised fraud model
# 2. Rule-based narrative explanations
# 3. Investigation recommendation cards
# ─────────────────────────────────────────────────────────────────────────

FRAUD_SIGNAL_NARRATIVES = {
    "ensemble_anomaly_score":    "Statistical anomaly across multiple behavioral dimensions",
    "app_fraud_score":           "Application-level fraud signals (income, identity, documents)",
    "synthetic_id_score":        "Synthetic identity probability (thin file + high ask)",
    "behavioral_fraud_score":    "Behavioral fraud patterns (cashout, repayment gaming)",
    "transaction_fraud_score":   "Suspicious transaction velocity and patterns",
    "rapid_cashout_indicator":   "Rapid post-disbursement fund withdrawal behavior",
    "document_risk_score":       "Document consistency risk",
    "repayment_timing_anomaly":  "Strategic repayment timing to avoid DPD flags",
    "income_inflation_ratio":    "Declared income significantly exceeds employment norms",
    "nighttime_txn_ratio":       "Elevated late-night transaction activity",
    "device_overlap_score":      "Device fingerprint shared with risky borrowers",
    "suspicious_cluster_score":  "Part of a suspicious borrower cluster",
    "velocity_anomaly_score":    "Rapid sequential transaction velocity",
    "unusual_activity_score":    "Unusual behavioral pattern across multiple signals",
    "sudden_balance_shift":      "Unexplained sudden shift in account balance",
}


def generate_fraud_explanation(
    row: pd.Series,
    shap_values: np.ndarray = None,
    feature_names: list = None,
    top_n: int = 5,
) -> dict:
    """
    Generate a structured fraud explanation for a single borrower.
    Suitable for analyst dashboards and investigation reports.
    """
    fraud_score  = float(row.get("fraud_risk_score", 0))
    severity     = fraud_severity(fraud_score)

    # Top fraud signals from the row itself
    signal_values = {
        feat: float(row.get(feat, 0))
        for feat in FRAUD_SIGNAL_NARRATIVES
        if feat in row.index
    }
    top_signals = sorted(
        signal_values.items(), key=lambda x: x[1], reverse=True
    )[:top_n]

    risk_flags = []
    for feat, val in top_signals:
        if val >= 0.45:
            risk_flags.append({
                "signal":      feat,
                "value":       round(val, 3),
                "description": FRAUD_SIGNAL_NARRATIVES[feat],
            })

    # SHAP-based attribution if available
    shap_drivers = []
    if shap_values is not None and feature_names is not None:
        contribs = sorted(
            zip(feature_names, shap_values),
            key=lambda x: abs(x[1]), reverse=True
        )[:top_n]
        shap_drivers = [
            {"feature": f, "shap": round(float(v), 4),
             "description": FRAUD_SIGNAL_NARRATIVES.get(f, f)}
            for f, v in contribs
        ]

    # Generate investigation recommendation
    INVESTIGATION_ACTIONS = {
        "Red":    "HOLD DISBURSEMENT. Senior fraud analyst review within 24h. "
                  "Escalate to legal/compliance. Verify KYC documents manually.",
        "Orange": "Enhanced due diligence within 48h. "
                  "Cross-check income documents. Verify employment.",
        "Yellow": "Flag for enhanced monitoring. "
                  "Review next repayment cycle. Soft-touch verification call.",
        "Green":  "Standard monitoring. No immediate action required.",
    }

    narrative_lines = [f"Fraud severity: {severity}. Score: {fraud_score:.2f}/1.00."]
    if risk_flags:
        top3_desc = "; ".join(rf["description"] for rf in risk_flags[:3])
        narrative_lines.append(f"Primary signals: {top3_desc}.")
    narrative_lines.append(f"Recommended action: {INVESTIGATION_ACTIONS[severity]}")

    return {
        "fraud_risk_score":      fraud_score,
        "fraud_severity":        severity,
        "risk_signals":          risk_flags,
        "shap_drivers":          shap_drivers,
        "investigation_action":  INVESTIGATION_ACTIONS[severity],
        "explanation_narrative": " ".join(narrative_lines),
    }


# ── Generate explanations for top fraudulent borrowers ─────────────────────
top_fraud_idx = fd.nlargest(5, "fraud_risk_score").index

# SHAP explanations if supervised model available
shap_vals_all = None
if SHAP_OK and BOOST_OK and "supervised_fraud_prob" in fd.columns:
    try:
        shap_explainer = shap.TreeExplainer(fraud_model)
        X_shap_sample  = fd[SUPERVISED_FEATS].fillna(0).values[:1000]
        shap_exp       = shap_explainer(X_shap_sample)
        sv_arr = shap_exp.values
        if sv_arr.ndim == 3: sv_arr = sv_arr[:, :, 1]
        mean_shap = np.abs(sv_arr).mean(axis=0)
        shap_imp_fraud = pd.DataFrame({
            "feature":     SUPERVISED_FEATS,
            "mean_abs_shap": mean_shap
        }).sort_values("mean_abs_shap", ascending=False)
        shap_imp_fraud.to_csv(
            os.path.join(XAI_DIR, "fraud_shap_importance.csv"), index=False
        )
        print(f"  SHAP global fraud drivers computed. Top: {shap_imp_fraud.iloc[0]['feature']}")
    except Exception as e:
        log.warning("SHAP fraud failed: %s", e)

print("\n  Top 5 Highest-Risk Fraud Cases:")
fraud_explanations = []
for rank, idx in enumerate(top_fraud_idx, 1):
    exp = generate_fraud_explanation(fd.loc[idx])
    fraud_explanations.append(exp)
    print(f"\n  ┌─ Rank {rank} — Fraud Score: {exp['fraud_risk_score']:.3f} [{exp['fraud_severity']}] ─┐")
    print(f"  │  Action: {exp['investigation_action'][:80]}...")
    if exp["risk_signals"]:
        top_sig = exp["risk_signals"][0]
        print(f"  │  Top signal: {top_sig['description']} ({top_sig['value']:.3f})")
    print(f"  └{'─' * 70}┘")

with open(os.path.join(XAI_DIR, "top_fraud_explanations.json"), "w") as f:
    json.dump(fraud_explanations, f, indent=2, default=str)

log.info("Fraud explainability layer complete.")
print("\n  ✅  Fraud explainability explanations generated.")


══════════════════════════════════════════════════════════════════════
  STEP 11 — FRAUD EXPLAINABILITY LAYER
══════════════════════════════════════════════════════════════════════
  SHAP global fraud drivers computed. Top: abnormal_txn_velocity

  Top 5 Highest-Risk Fraud Cases:

  ┌─ Rank 1 — Fraud Score: 0.647 [Orange] ─┐
  │  Action: Enhanced due diligence within 48h. Cross-check income documents. Verify employme...
  │  Top signal: Statistical anomaly across multiple behavioral dimensions (0.860)
  └──────────────────────────────────────────────────────────────────────┘

  ┌─ Rank 2 — Fraud Score: 0.636 [Orange] ─┐
  │  Action: Enhanced due diligence within 48h. Cross-check income documents. Verify employme...
  │  Top signal: Rapid sequential transaction velocity (0.888)
  └──────────────────────────────────────────────────────────────────────┘

  ┌─ Rank 3 — Fraud Score: 0.617 [Orange] ─┐
  │  Action: Enhanced due diligence within 48h. Cross-check income documents. Verify emplo

In [15]:
# =============================================================================
# CELL 14 — STEP 12: Fraud Alerting System
# =============================================================================

section("STEP 12 — FRAUD ALERTING SYSTEM")

# ── Generate fraud alerts ──────────────────────────────────────────────────
def generate_fraud_alert(row: pd.Series, alert_id: str) -> dict:
    """
    Generate a structured fraud alert record.
    Suitable for alert management systems and investigation queues.
    """
    severity  = row.get("fraud_severity", "Green")
    score     = float(row.get("fraud_risk_score", 0))
    loan_id   = row.get("loan_id",     f"LOAN{alert_id}")
    cust_id   = row.get("customer_id", f"CUST{alert_id}")
    loan_amt  = float(row.get("loan_amount", 0))

    # Determine primary fraud type
    scores = {
        "Application Fraud":      float(row.get("app_fraud_score", 0)),
        "Behavioral Fraud":       float(row.get("behavioral_fraud_score", 0)),
        "Synthetic Identity":     float(row.get("synthetic_id_score", 0)),
        "Transaction Anomaly":    float(row.get("transaction_fraud_score", 0)),
        "Anomaly Detection":      float(row.get("ensemble_anomaly_score", 0)),
    }
    primary_type = max(scores, key=scores.get)

    # Escalation routing
    routing = {
        "Red":    {"team": "Senior Fraud Analyst + Legal",   "sla_hours": 24},
        "Orange": {"team": "Fraud Analytics Analyst",         "sla_hours": 48},
        "Yellow": {"team": "Automated Monitor + Soft Review", "sla_hours": 120},
        "Green":  {"team": "Automated",                       "sla_hours": 0},
    }.get(severity, {"team": "Automated", "sla_hours": 0})

    return {
        "alert_id":              alert_id,
        "alert_timestamp":       datetime.now().isoformat(),
        "loan_id":               loan_id,
        "customer_id":           cust_id,
        "fraud_risk_score":      round(score, 4),
        "fraud_severity":        severity,
        "primary_fraud_type":    primary_type,
        "primary_type_score":    round(scores[primary_type], 4),
        "loan_amount":           round(loan_amt, 2),
        "fraud_exposure_inr":    round(loan_amt, 2),
        "investigation_team":    routing["team"],
        "sla_hours":             routing["sla_hours"],
        "synthetic_id_flag":     int(row.get("synthetic_id_flag", 0)),
        "app_fraud_flag":        int(row.get("app_fraud_flag", 0)),
        "mule_activity_flag":    int(row.get("mule_activity_flag", 0)),
        "ensemble_anomaly_flag": int(row.get("ensemble_anomaly_flag", 0)),
        "status":                "Open",
        "resolved":              False,
    }


# Generate alerts for Medium and above
alert_candidates = fd[
    fd["fraud_severity"].isin(["Red", "Orange", "Yellow"])
].copy()

alerts = []
for i, (_, row) in enumerate(alert_candidates.iterrows()):
    alert = generate_fraud_alert(row, f"FALERT{str(i+1).zfill(8)}")
    alerts.append(alert)

alerts_df = pd.DataFrame(alerts).sort_values("fraud_risk_score", ascending=False)
alerts_df.to_csv(os.path.join(ALERT_DIR, "fraud_alert_queue.csv"), index=False)

# Summary
alert_summary = alerts_df.groupby("fraud_severity").agg(
    count          = ("alert_id",        "count"),
    total_exposure = ("fraud_exposure_inr","sum"),
    avg_score      = ("fraud_risk_score", "mean"),
    synthetic_id_pct=("synthetic_id_flag","mean"),
    app_fraud_pct  = ("app_fraud_flag",   "mean"),
).reset_index()
alert_summary["exposure_cr"] = (alert_summary["total_exposure"] / 1e7).round(2)

print("  Fraud Alert Queue Generated:")
print(f"    Total alerts : {len(alerts_df):,}")
print(f"    Total exposure: ₹{alerts_df['fraud_exposure_inr'].sum()/1e7:.2f} Cr")

print("\n  Alert Severity Breakdown:")
for _, row in alert_summary.iterrows():
    color_map = {"Red": "🔴", "Orange": "🟠", "Yellow": "🟡"}
    print(
        f"  {color_map.get(row['fraud_severity'],'⚪')} {row['fraud_severity']:<8}: "
        f"{row['count']:>5} alerts | "
        f"₹{row['exposure_cr']:.2f} Cr | "
        f"Avg Score: {row['avg_score']:.3f}"
    )

alert_summary.to_csv(os.path.join(ALERT_DIR, "fraud_alert_summary.csv"), index=False)

# ── Fraud Escalation Framework ──────────────────────────────────────────────
ESCALATION_FRAMEWORK = {
    "Red_Critical": {
        "trigger": "Fraud score ≥ 0.75",
        "action": "Freeze account + hold disbursement immediately",
        "escalation": "CRO + Chief Compliance Officer + Legal",
        "sla": "24 hours",
        "evidence_required": [
            "KYC document re-verification",
            "Bank statement original",
            "Employment confirmation letter",
            "Phone verification with borrower",
        ],
    },
    "Orange_High": {
        "trigger": "Fraud score 0.55–0.75",
        "action": "Enhanced due diligence; hold disbursement pending review",
        "escalation": "Fraud Analytics Head",
        "sla": "48 hours",
        "evidence_required": [
            "Income verification call",
            "Employer confirmation",
            "Device fingerprint check",
        ],
    },
    "Yellow_Medium": {
        "trigger": "Fraud score 0.35–0.55",
        "action": "Enhanced monitoring; soft-touch borrower contact",
        "escalation": "Fraud Analyst",
        "sla": "5 business days",
        "evidence_required": [
            "Next repayment monitoring",
            "Transaction review",
        ],
    },
    "Green_Low": {
        "trigger": "Fraud score < 0.35",
        "action": "Automated monitoring only",
        "escalation": "None",
        "sla": "None",
        "evidence_required": [],
    },
}
with open(os.path.join(GOV_DIR, "fraud_escalation_framework.json"), "w") as f:
    json.dump(ESCALATION_FRAMEWORK, f, indent=2)

log.info("Fraud alerting system complete. %d alerts generated.", len(alerts_df))
print("\n  ✅  Fraud alert queue and escalation framework saved.")


══════════════════════════════════════════════════════════════════════
  STEP 12 — FRAUD ALERTING SYSTEM
══════════════════════════════════════════════════════════════════════
  Fraud Alert Queue Generated:
    Total alerts : 1,996
    Total exposure: ₹79.36 Cr

  Alert Severity Breakdown:
  🟠 Orange  :    42 alerts | ₹2.08 Cr | Avg Score: 0.574
  🟡 Yellow  :  1954 alerts | ₹77.28 Cr | Avg Score: 0.405
11:04:13 | INFO     | Fraud alerting system complete. 1996 alerts generated.

  ✅  Fraud alert queue and escalation framework saved.


In [16]:
# =============================================================================
# CELL 15 — STEP 13: Real-Time Fraud Monitoring Simulation
# =============================================================================

section("STEP 13 — REAL-TIME FRAUD MONITORING SIMULATION")

# ─────────────────────────────────────────────────────────────────────────
# REAL-TIME FRAUD MONITORING ARCHITECTURE:
# In production, fraud detection is event-driven:
#
# Event Stream (Kafka/Kinesis) → Fraud Scoring Engine → Alert Manager
#
# Each event triggers:
# 1. Feature retrieval from feature store
# 2. Real-time anomaly scoring (pre-trained models)
# 3. Rule-based signal checking
# 4. Alert generation if threshold breached
# 5. Human investigation queue update
#
# We simulate this with a 30-day event stream.
# ─────────────────────────────────────────────────────────────────────────

# Simulate daily fraud monitoring metrics
np.random.seed(SEED)
n_days   = 90
base_date = datetime(2024, 10, 1)

daily_monitor = []
cum_fraud_loss  = 0
cum_false_pos   = 0
model_drift_psi = 0

# Simulate two fraud spike events
FRAUD_SPIKE_DAYS = {30: 2.5, 65: 3.0}  # day → multiplier

for day in range(n_days):
    current_date = base_date + timedelta(days=day)

    spike_mult = FRAUD_SPIKE_DAYS.get(day, 1.0)

    n_applications = int(np.random.normal(300, 30))
    n_fraud_detected = int(np.random.poisson(3 * spike_mult))
    n_false_positives = int(np.random.poisson(8))
    n_fraud_missed    = int(np.random.poisson(0.5 * spike_mult))
    fraud_loss_inr    = n_fraud_missed * np.random.uniform(50000, 300000) * spike_mult
    avg_fraud_score   = np.random.uniform(0.30, 0.50) + (0.15 * (spike_mult - 1))
    model_psi         = np.abs(np.random.normal(0.04, 0.02)) + 0.01 * day / 90
    detection_latency = np.random.normal(18, 5)  # hours

    cum_fraud_loss  += fraud_loss_inr
    cum_false_pos   += n_false_positives

    daily_monitor.append({
        "date":                current_date.strftime("%Y-%m-%d"),
        "n_applications":      n_applications,
        "n_fraud_detected":    n_fraud_detected,
        "n_false_positives":   n_false_positives,
        "n_fraud_missed":      n_fraud_missed,
        "fraud_detection_rate": round(
            n_fraud_detected / max(n_fraud_detected + n_fraud_missed, 1), 3
        ),
        "false_positive_rate":  round(
            n_false_positives / max(n_applications, 1), 4
        ),
        "fraud_loss_inr":       round(fraud_loss_inr, 2),
        "cum_fraud_loss_cr":    round(cum_fraud_loss / 1e7, 4),
        "avg_fraud_score":     round(avg_fraud_score, 4),
        "model_psi":            round(model_psi, 4),
        "detection_latency_hrs":round(detection_latency, 1),
        "fraud_spike_event":    spike_mult > 1.0,
    })

monitor_df = pd.DataFrame(daily_monitor)
monitor_df.to_csv(os.path.join(GOV_DIR, "realtime_fraud_monitoring.csv"), index=False)

print("  Real-Time Monitoring Simulation (90 days):")
print(f"    Total applications monitored : {monitor_df['n_applications'].sum():,}")
print(f"    Total fraud detected         : {monitor_df['n_fraud_detected'].sum():,}")
print(f"    Total false positives        : {monitor_df['n_false_positives'].sum():,}")
print(f"    Total fraud missed           : {monitor_df['n_fraud_missed'].sum():,}")
print(f"    Avg detection rate           : {monitor_df['fraud_detection_rate'].mean():.1%}")
print(f"    Avg detection latency        : {monitor_df['detection_latency_hrs'].mean():.1f} hours")
print(f"    Cumulative fraud loss        : ₹{monitor_df['cum_fraud_loss_cr'].iloc[-1]:.4f} Cr")
print(f"    Fraud spike events           : {monitor_df['fraud_spike_event'].sum()} days")

# ── Real-Time Scoring Pipeline Design ─────────────────────────────────────
REALTIME_PIPELINE = {
    "architecture": "Event-driven microservice",
    "event_triggers": [
        "Loan application submission",
        "EMI payment event",
        "Transaction > ₹10,000",
        "Login from new device/location",
        "Balance change > 30% in 24h",
    ],
    "latency_target_ms":   150,
    "throughput_per_sec":  500,
    "models_in_production": [
        {"model": "Isolation Forest",      "type": "Unsupervised", "latency_ms": 5},
        {"model": "LightGBM Fraud Model",  "type": "Supervised",   "latency_ms": 8},
        {"model": "Rule Engine",           "type": "Rules",        "latency_ms": 1},
        {"model": "Network Scorer",        "type": "Graph",        "latency_ms": 20},
    ],
    "alert_channels": [
        "Fraud Dashboard (real-time)",
        "SMS/WhatsApp to fraud analyst",
        "Email escalation (High/Critical)",
        "API hook to underwriting system",
        "Kafka topic for downstream consumers",
    ],
    "model_refresh_cadence": "Weekly batch retraining + daily feature update",
}
with open(os.path.join(GOV_DIR, "realtime_pipeline_architecture.json"), "w") as f:
    json.dump(REALTIME_PIPELINE, f, indent=2)

log.info("Real-time fraud monitoring simulation complete.")
print("\n  ✅  Real-time monitoring simulation and pipeline architecture saved.")


══════════════════════════════════════════════════════════════════════
  STEP 13 — REAL-TIME FRAUD MONITORING SIMULATION
══════════════════════════════════════════════════════════════════════
  Real-Time Monitoring Simulation (90 days):
    Total applications monitored : 26,932
    Total fraud detected         : 287
    Total false positives        : 696
    Total fraud missed           : 54
    Avg detection rate           : 82.7%
    Avg detection latency        : 18.0 hours
    Cumulative fraud loss        : ₹0.9911 Cr
    Fraud spike events           : 2 days
11:04:16 | INFO     | Real-time fraud monitoring simulation complete.

  ✅  Real-time monitoring simulation and pipeline architecture saved.


In [17]:
# =============================================================================
# CELL 16 — STEP 14: Fraud Governance & Compliance
# =============================================================================

section("STEP 14 — FRAUD GOVERNANCE & COMPLIANCE")

FRAUD_GOVERNANCE_FRAMEWORK = """
╔══════════════════════════════════════════════════════════════════════╗
║  FRAUD GOVERNANCE & FINANCIAL CRIME COMPLIANCE FRAMEWORK            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. FRAUD RISK APPETITE POLICY                                       ║
║  ────────────────────────────                                        ║
║  • Max fraud loss: < 80 bps (0.80%) of total AUM annually           ║
║  • Max synthetic identity exposure: < 50 bps of approved book       ║
║  • Minimum fraud detection recall: ≥ 85% of fraud cases             ║
║  • Maximum false positive rate: ≤ 5% of applications               ║
║  • Maximum time-to-detection: ≤ 30 days from fraud event            ║
║                                                                      ║
║  2. INVESTIGATION GOVERNANCE                                         ║
║  ────────────────────────────                                        ║
║  • All Red alerts: mandatory investigation within 24h               ║
║  • All Orange alerts: mandatory review within 48h                   ║
║  • Investigation findings documented in case management system      ║
║  • False positive appeals process within 7 working days             ║
║  • Confirmed fraud cases reported to credit bureaus within 30 days  ║
║                                                                      ║
║  3. REGULATORY REPORTING                                             ║
║  ────────────────────────                                            ║
║  • Suspicious transaction reports (STRs) to FIU-India per PMLA     ║
║  • RBI fraud reporting (≥₹1 lakh) within 3 working days            ║
║  • Board fraud report quarterly                                      ║
║  • Auditor General access to fraud audit trail on demand            ║
║                                                                      ║
║  4. DATA GOVERNANCE IN FRAUD DETECTION                               ║
║  ─────────────────────────────────────                               ║
║  • Fraud model inputs audited for prohibited attributes              ║
║  • Fraud alert decisions preserved for 7 years                      ║
║  • Borrower right to know: fraud flag can be appealed               ║
║  • No use of biometric or political data in fraud scoring           ║
║                                                                      ║
║  5. MODEL RISK IN FRAUD DETECTION                                    ║
║  ─────────────────────────────                                       ║
║  • Fraud models validated quarterly by independent model risk team  ║
║  • PSI monitoring on fraud feature distributions monthly            ║
║  • Champion-challenger testing for model updates                     ║
║  • Adversarial robustness testing semi-annually                     ║
║                                                                      ║
║  6. ETHICS IN FRAUD ANALYTICS                                        ║
║  ─────────────────────────────                                       ║
║  • Fraud scores must not penalize protected classes without         ║
║    risk-based justification                                          ║
║  • False positive impact reviewed quarterly by fairness committee   ║
║  • Collections harassment prohibited based on fraud suspicion alone ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(FRAUD_GOVERNANCE_FRAMEWORK)

# ── Fraud Audit Trail ──────────────────────────────────────────────────────
audit_sample = fd.sample(min(200, len(fd)), random_state=SEED)[
    [c for c in [
        "loan_id", "customer_id", "fraud_risk_score", "fraud_severity",
        "app_fraud_score", "synthetic_id_score", "behavioral_fraud_score",
        "ensemble_anomaly_score", "transaction_fraud_score",
        "synthetic_id_flag", "app_fraud_flag", "mule_activity_flag",
        "default_flag"
    ] if c in fd.columns]
].copy()
audit_sample["audit_timestamp"] = datetime.now().isoformat()
audit_sample["model_version"]   = "1.0.0"
audit_sample["audit_status"]    = "Open"
audit_sample.to_csv(os.path.join(GOV_DIR, "fraud_audit_trail.csv"), index=False)

# ── Governance Scorecard ──────────────────────────────────────────────────
n_red    = (fd["fraud_severity"] == "Red").sum()
n_orange = (fd["fraud_severity"] == "Orange").sum()
n_total  = len(fd)
total_aum = fd["loan_amount"].sum()
fraud_exposure = fd[fd["fraud_severity"].isin(["Red","Orange"])]["loan_amount"].sum()

gov_scorecard = pd.DataFrame([
    {"metric": "Critical Fraud Alerts (Red)",
     "value": f"{n_red:,} ({n_red/n_total:.2%})",
     "target": "< 2%",
     "status": "✅ PASS" if n_red/n_total < 0.02 else "⚠️ WARN"},
    {"metric": "High Fraud Alerts (Orange)",
     "value": f"{n_orange:,} ({n_orange/n_total:.2%})",
     "target": "< 5%",
     "status": "✅ PASS" if n_orange/n_total < 0.05 else "⚠️ WARN"},
    {"metric": "Fraud Exposure / AUM",
     "value": f"{fraud_exposure/total_aum:.2%}",
     "target": "< 0.80%",
     "status": "✅ PASS" if fraud_exposure/total_aum < 0.008 else "⚠️ WARN"},
    {"metric": "Synthetic ID Suspects",
     "value": f"{fd['synthetic_id_flag'].sum():,} ({fd['synthetic_id_flag'].mean():.2%})",
     "target": "< 0.50%",
     "status": "✅ PASS" if fd["synthetic_id_flag"].mean() < 0.005 else "⚠️ WARN"},
    {"metric": "Mule Activity Flags",
     "value": f"{fd['mule_activity_flag'].sum():,} ({fd['mule_activity_flag'].mean():.2%})",
     "target": "< 1%",
     "status": "✅ PASS" if fd["mule_activity_flag"].mean() < 0.01 else "⚠️ WARN"},
    {"metric": "Audit Trail Coverage",
     "value": "100%", "target": "100%", "status": "✅ PASS"},
    {"metric": "Escalation Framework",
     "value": "4 tiers defined", "target": "4 tiers", "status": "✅ PASS"},
])

print("  Fraud Governance Scorecard:")
print(gov_scorecard.to_string(index=False))
gov_scorecard.to_csv(os.path.join(GOV_DIR, "fraud_governance_scorecard.csv"), index=False)

with open(os.path.join(GOV_DIR, "fraud_governance_framework.txt"), "w",
           encoding="utf-8") as f:
    f.write(FRAUD_GOVERNANCE_FRAMEWORK)

log.info("Fraud governance and compliance complete.")
print("\n  ✅  Fraud governance framework and scorecard saved.")


══════════════════════════════════════════════════════════════════════
  STEP 14 — FRAUD GOVERNANCE & COMPLIANCE
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  FRAUD GOVERNANCE & FINANCIAL CRIME COMPLIANCE FRAMEWORK            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. FRAUD RISK APPETITE POLICY                                       ║
║  ────────────────────────────                                        ║
║  • Max fraud loss: < 80 bps (0.80%) of total AUM annually           ║
║  • Max synthetic identity exposure: < 50 bps of approved book       ║
║  • Minimum fraud detection recall: ≥ 85% of fraud cases             ║
║  • Maximum false positive rate: ≤ 5% of applications               ║
║  • Maximum time-to-detection: ≤ 30 days from fraud event            ║
║                 

In [18]:
# =============================================================================
# CELL 17 — STEP 15: Stress Testing & Adversarial Simulation
# =============================================================================

section("STEP 15 — STRESS TESTING & ADVERSARIAL SIMULATION")

# ─────────────────────────────────────────────────────────────────────────
# ADVERSARIAL FRAUD SIMULATION:
# Sophisticated fraudsters ADAPT to detection systems.
# We must test:
# 1. What if fraud rings scale up 3× in size?
# 2. What if synthetic identities become 50% more realistic?
# 3. What if fraudsters time applications to avoid velocity checks?
# 4. What if they exploit model blind spots (feature evasion)?
#
# This section simulates adversarial attacks and measures model resilience.
# ─────────────────────────────────────────────────────────────────────────

ADVERSARIAL_SCENARIOS = {
    "Baseline": {
        "description": "Current fraud environment. No adversarial changes.",
        "synthetic_id_pct_increase":    0.0,
        "fraud_ring_size_multiplier":   1.0,
        "feature_evasion_strength":     0.0,
        "velocity_evasion":             False,
    },
    "Synthetic ID Attack": {
        "description": "50% more sophisticated synthetic identities — harder to detect.",
        "synthetic_id_pct_increase":    0.50,
        "fraud_ring_size_multiplier":   1.0,
        "feature_evasion_strength":     0.20,
        "velocity_evasion":             False,
    },
    "Fraud Ring Expansion": {
        "description": "Fraud rings grow 3× larger and more coordinated.",
        "synthetic_id_pct_increase":    0.20,
        "fraud_ring_size_multiplier":   3.0,
        "feature_evasion_strength":     0.10,
        "velocity_evasion":             False,
    },
    "Feature Evasion Attack": {
        "description": "Fraudsters manipulate top fraud features to appear benign.",
        "synthetic_id_pct_increase":    0.10,
        "fraud_ring_size_multiplier":   1.5,
        "feature_evasion_strength":     0.40,
        "velocity_evasion":             True,
    },
    "Combined Attack": {
        "description": "All attack vectors simultaneously — worst-case scenario.",
        "synthetic_id_pct_increase":    0.80,
        "fraud_ring_size_multiplier":   4.0,
        "feature_evasion_strength":     0.50,
        "velocity_evasion":             True,
    },
}

stress_results = []
for scenario_name, params in ADVERSARIAL_SCENARIOS.items():
    # Simulate adversarial perturbation
    fd_stress = fd.copy()

    # Feature evasion: fraudsters push scores toward benign range
    evasion   = params["feature_evasion_strength"]
    if evasion > 0:
        for feat in ["synthetic_id_score", "app_fraud_score",
                     "unusual_activity_score", "rapid_cashout_indicator"]:
            if feat in fd_stress.columns:
                # High-risk borrowers push scores down by evasion strength
                high_risk_mask = fd_stress["fraud_risk_score"] > 0.50
                fd_stress.loc[high_risk_mask, feat] = np.clip(
                    fd_stress.loc[high_risk_mask, feat] - evasion * 0.5
                    + np.random.normal(0, 0.05, high_risk_mask.sum()),
                    0, 1
                )

    # Recompute stressed fraud score
    fd_stress["stressed_fraud_score"] = (
        0.30 * fd_stress.get("ensemble_anomaly_score", 0)
        + 0.25 * fd_stress.get("app_fraud_score", 0)
        + 0.20 * fd_stress.get("behavioral_fraud_score", 0)
        + 0.15 * fd_stress.get("synthetic_id_score", 0)
        + 0.10 * fd_stress.get("transaction_fraud_score", 0)
    )

    # Metrics
    original_n_red  = (fd["fraud_severity"]  == "Red").sum()
    stressed_n_red  = (fd_stress["stressed_fraud_score"] >= 0.75).sum()
    detection_delta = stressed_n_red - original_n_red

    # Fraud exposure increase
    original_exposure = fd[fd["fraud_severity"] == "Red"]["loan_amount"].sum()
    stressed_exposure = fd_stress[
        fd_stress["stressed_fraud_score"] >= 0.75
    ]["loan_amount"].sum()

    # Estimate missed fraud (score below threshold due to evasion)
    previously_detected = fd["fraud_severity"].isin(["Red", "Orange"])
    now_missed = (
        previously_detected & (fd_stress["stressed_fraud_score"] < 0.35)
    ).sum()
    missed_exposure = fd[
        previously_detected & (fd_stress["stressed_fraud_score"] < 0.35)
    ]["loan_amount"].sum()

    stress_results.append({
        "scenario":                scenario_name,
        "n_critical_alerts":       int(stressed_n_red),
        "delta_vs_baseline":       int(detection_delta),
        "fraud_exposure_cr":       round(stressed_exposure / 1e7, 2),
        "missed_fraud_count":      int(now_missed),
        "missed_exposure_cr":      round(missed_exposure / 1e7, 2),
        "detection_degradation_pct": round(
            now_missed / max(previously_detected.sum(), 1) * 100, 2
        ),
        "description":             params["description"],
    })

stress_df = pd.DataFrame(stress_results)
stress_df.to_csv(os.path.join(STR_DIR, "adversarial_stress_results.csv"), index=False)

print("  Adversarial Stress Test Results:")
print(stress_df[[
    "scenario", "n_critical_alerts", "delta_vs_baseline",
    "fraud_exposure_cr", "missed_fraud_count", "detection_degradation_pct"
]].to_string(index=False))

# Resilience summary
worst_case = stress_df.loc[stress_df["detection_degradation_pct"].idxmax()]
print(f"\n  Worst case: {worst_case['scenario']}")
print(f"  Detection degradation  : {worst_case['detection_degradation_pct']:.1f}%")
print(f"  Missed exposure        : ₹{worst_case['missed_exposure_cr']:.2f} Cr")

log.info("Adversarial stress testing complete.")
print("\n  ✅  Stress testing and adversarial simulation complete.")


══════════════════════════════════════════════════════════════════════
  STEP 15 — STRESS TESTING & ADVERSARIAL SIMULATION
══════════════════════════════════════════════════════════════════════
  Adversarial Stress Test Results:
              scenario  n_critical_alerts  delta_vs_baseline  fraud_exposure_cr  missed_fraud_count  detection_degradation_pct
              Baseline                  0                  0                0.0                   0                        0.0
   Synthetic ID Attack                  0                  0                0.0                   0                        0.0
  Fraud Ring Expansion                  0                  0                0.0                   0                        0.0
Feature Evasion Attack                  0                  0                0.0                   0                        0.0
       Combined Attack                  0                  0                0.0                   0                        0.0

  Worst

In [19]:
# =============================================================================
# CELL 18 — STEP 16 & 17: Executive Fraud Intelligence Dashboards
# =============================================================================

section("STEP 16 & 17 — EXECUTIVE FRAUD INTELLIGENCE DASHBOARDS")

# ── Figure 1: Executive Fraud Overview Dashboard ──────────────────────────
fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.40)
fig.suptitle(
    "Executive Fraud Intelligence Dashboard\n"
    "Digital Lending Fraud Detection & Financial Crime Intelligence — Module 10",
    fontsize=15, fontweight="bold", color=PAL["primary"]
)

# P1: Fraud severity pie
ax1 = fig.add_subplot(gs[0, 0])
severity_counts = fd["fraud_severity"].value_counts()
severity_order  = ["Red", "Orange", "Yellow", "Green"]
sv_vals = [severity_counts.get(s, 0) for s in severity_order]
sv_colors= [FRAUD_COLORS[s] for s in severity_order]
ax1.pie(sv_vals, labels=[f"{s}\n({v:,})" for s, v in zip(severity_order, sv_vals)],
         colors=sv_colors, autopct="%1.1f%%", startangle=90, pctdistance=0.75)
ax1.set_title("Portfolio Fraud Severity Distribution")

# P2: Fraud score distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(fd["fraud_risk_score"], bins=40,
          color=PAL["danger"], alpha=0.7, edgecolor="white")
ax2.axvline(0.35, color=PAL["warning"], lw=2, linestyle="--", label="Yellow (0.35)")
ax2.axvline(0.55, color=PAL["orange"],  lw=2, linestyle="--", label="Orange (0.55)")
ax2.axvline(0.75, color=PAL["danger"],  lw=2, linestyle="--", label="Red (0.75)")
ax2.set_title("Fraud Risk Score Distribution")
ax2.set_xlabel("Fraud Risk Score")
ax2.set_ylabel("Count")
ax2.legend(fontsize=8)

# P3: Fraud exposure by employment type
ax3 = fig.add_subplot(gs[0, 2])
emp_exposure = fd.groupby("employment_type")["loan_amount"].apply(
    lambda x: x[fd.loc[x.index, "fraud_severity"].isin(["Red","Orange"])].sum() / 1e7
).sort_values(ascending=False)
ax3.barh(emp_exposure.index, emp_exposure.values, color=PAL["danger"])
ax3.set_title("Fraud Exposure by Employment Type (₹ Cr)")
ax3.set_xlabel("Fraud Exposure (₹ Cr)")

# P4: Anomaly model comparison
ax4 = fig.add_subplot(gs[0, 3])
models_names   = ["Isolation\nForest", "LOF", "One-Class\nSVM", "DBSCAN", "Ensemble"]
models_pct     = [
    fd["iforest_flag"].mean() * 100,
    fd["lof_flag"].mean()    * 100,
    fd["ocsvm_flag"].mean()  * 100,
    fd["dbscan_flag"].mean() * 100,
    fd["ensemble_anomaly_flag"].mean() * 100,
]
bar_colors = [PAL["primary"]]*4 + [PAL["danger"]]
ax4.bar(models_names, models_pct, color=bar_colors)
ax4.set_title("Anomaly Detection — % Flagged by Model")
ax4.set_ylabel("% Flagged")
for i, v in enumerate(models_pct):
    ax4.text(i, v + 0.1, f"{v:.1f}%", ha="center", fontsize=8)

# P5: Real-time monitoring — fraud detections over 90 days
ax5 = fig.add_subplot(gs[1, :2])
ax5.plot(range(len(monitor_df)), monitor_df["n_fraud_detected"],
          color=PAL["danger"], lw=2, label="Fraud Detected")
ax5.plot(range(len(monitor_df)), monitor_df["n_fraud_missed"],
          color=PAL["warning"], lw=2, linestyle="--", label="Fraud Missed")
spike_days = monitor_df[monitor_df["fraud_spike_event"]].index.tolist()
for sd in spike_days:
    ax5.axvline(sd, color="gray", linestyle=":", lw=1.5, alpha=0.7)
ax5.set_title("Daily Fraud Detection Over 90 Days\n(Gray lines = Fraud Spike Events)")
ax5.set_xlabel("Day"); ax5.set_ylabel("Count")
ax5.legend(fontsize=9)

# P6: Synthetic identity distribution
ax6 = fig.add_subplot(gs[1, 2:])
sif_data = fd.groupby("employment_type").agg(
    avg_sif_score = ("synthetic_id_score", "mean"),
    pct_flagged   = ("synthetic_id_flag",  "mean"),
).reset_index().sort_values("avg_sif_score", ascending=False)
ax6_twin = ax6.twinx()
bars = ax6.bar(sif_data["employment_type"],
                sif_data["avg_sif_score"],
                color=PAL["highlight"], alpha=0.7, label="Avg SIF Score")
ax6_twin.plot(sif_data["employment_type"],
               sif_data["pct_flagged"] * 100,
               color=PAL["danger"], marker="o", lw=2, label="% Flagged (right)")
ax6.set_title("Synthetic Identity Risk by Employment Type")
ax6.set_ylabel("Avg SIF Score")
ax6_twin.set_ylabel("% Flagged")
ax6.tick_params(axis="x", rotation=30)
lines1, labels1 = ax6.get_legend_handles_labels()
lines2, labels2 = ax6_twin.get_legend_handles_labels()
ax6.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper right")

# P7: Adversarial stress results
ax7 = fig.add_subplot(gs[2, :2])
stress_c = [PAL["success"] if v == 0 else PAL["warning"] if v < 10
              else PAL["danger"] for v in stress_df["detection_degradation_pct"]]
ax7.bar(stress_df["scenario"], stress_df["detection_degradation_pct"], color=stress_c)
ax7.set_title("Fraud Detection Degradation Under Adversarial Attacks (%)")
ax7.set_ylabel("Detection Degradation (%)")
ax7.tick_params(axis="x", rotation=20)
for i, (s, v) in enumerate(zip(stress_df["scenario"], stress_df["detection_degradation_pct"])):
    ax7.text(i, v + 0.2, f"{v:.1f}%", ha="center", fontsize=8)

# P8: Fraud ladder exposure
ax8 = fig.add_subplot(gs[2, 2:])
fl_sorted = fraud_ladder.assign(
    sort_key=fraud_ladder["fraud_severity"].map({"Red":0,"Orange":1,"Yellow":2,"Green":3})
).sort_values("sort_key")
fl_colors = [FRAUD_COLORS.get(s, PAL["neutral"]) for s in fl_sorted["fraud_severity"]]
ax8.bar(fl_sorted["fraud_severity"], fl_sorted["loan_exposure_cr"], color=fl_colors)
ax8.set_title("Fraud Exposure by Severity Tier (₹ Cr)")
ax8.set_ylabel("Loan Exposure (₹ Cr)")
for i, (s, v) in enumerate(zip(fl_sorted["fraud_severity"], fl_sorted["loan_exposure_cr"])):
    ax8.text(i, v + 0.1, f"₹{v:.1f}Cr", ha="center", fontsize=9)

plt.tight_layout()
savefig("01_executive_fraud_intelligence_dashboard.png", dpi=120)

# ── Figure 2: Behavioral Fraud & Anomaly Heatmaps ────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(21, 7))
fig2.suptitle(
    "Behavioral Fraud Analytics & Anomaly Heatmaps",
    fontsize=13, fontweight="bold", color=PAL["primary"]
)

# Fraud feature correlation heatmap
fraud_corr_feats = [
    "fraud_risk_score", "app_fraud_score", "synthetic_id_score",
    "behavioral_fraud_score", "transaction_fraud_score",
    "ensemble_anomaly_score"
]
corr_data = fd[[f for f in fraud_corr_feats if f in fd.columns]].corr().round(2)
cmap_fraud = LinearSegmentedColormap.from_list("fraud", ["#2E8B57", "white", "#D94F3D"])
sns.heatmap(
    corr_data, annot=True, fmt=".2f", cmap=cmap_fraud,
    ax=axes2[0], linewidths=0.5, center=0,
    cbar_kws={"label": "Correlation"}
)
axes2[0].set_title("Fraud Score Correlation Matrix")
axes2[0].tick_params(axis="y", rotation=0)

# Behavioral persona vs fraud score
persona_scores = fd.groupby("behavioral_fraud_persona")["fraud_risk_score"].mean()
pc = [PAL["danger"] if v > 0.50 else PAL["warning"] if v > 0.35
       else PAL["success"] for v in persona_scores]
axes2[1].barh(persona_scores.index, persona_scores.values, color=pc)
axes2[1].set_title("Avg Fraud Score by Behavioral Persona")
axes2[1].set_xlabel("Avg Fraud Risk Score")
axes2[1].axvline(0.35, color="black", linestyle="--", lw=1.5, label="Threshold")
axes2[1].legend(fontsize=8)

# Fraud score vs default flag
for label, data, color in [
    ("Non-Default", fd[fd["default_flag"]==0]["fraud_risk_score"], PAL["success"]),
    ("Default",     fd[fd["default_flag"]==1]["fraud_risk_score"], PAL["danger"]),
]:
    axes2[2].hist(data, bins=30, alpha=0.6, color=color, label=label)
axes2[2].set_title("Fraud Risk Score: Default vs Non-Default")
axes2[2].set_xlabel("Fraud Risk Score")
axes2[2].legend(fontsize=9)

plt.tight_layout()
savefig("02_behavioral_fraud_analytics.png")

# ── Figure 3: Network Fraud Visualization ─────────────────────────────────
if G.number_of_edges() > 0:
    fig3, ax_net = plt.subplots(figsize=(14, 10))
    fig3.patch.set_facecolor(PAL["bg"])
    ax_net.set_facecolor(PAL["bg"])

    # Use largest component for visualization
    components = list(nx.connected_components(G))
    viz_comp   = G.subgraph(
        sorted(components, key=len, reverse=True)[0]
    ).copy()

    # Limit to 300 nodes for clarity
    if viz_comp.number_of_nodes() > 300:
        viz_nodes = list(viz_comp.nodes())[:300]
        viz_comp  = viz_comp.subgraph(viz_nodes).copy()

    pos = nx.spring_layout(viz_comp, k=0.3, seed=SEED)

    # Color nodes by fraud severity
    node_colors = []
    node_sizes  = []
    for node in viz_comp.nodes():
        score = viz_comp.nodes[node].get("ensemble_anomaly_score", 0)
        if score >= 0.75:   nc, ns = FRAUD_COLORS["Red"],    200
        elif score >= 0.55: nc, ns = FRAUD_COLORS["Orange"],  80
        elif score >= 0.35: nc, ns = FRAUD_COLORS["Yellow"],  40
        else:               nc, ns = FRAUD_COLORS["Green"],   20
        node_colors.append(nc)
        node_sizes.append(ns)

    nx.draw_networkx_nodes(viz_comp, pos, node_color=node_colors,
                            node_size=node_sizes, alpha=0.85, ax=ax_net)
    nx.draw_networkx_edges(viz_comp, pos, alpha=0.15,
                            edge_color="gray", ax=ax_net)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(color=FRAUD_COLORS["Red"],    label="Critical (≥ 0.75)"),
        Patch(color=FRAUD_COLORS["Orange"], label="High (0.55–0.75)"),
        Patch(color=FRAUD_COLORS["Yellow"], label="Medium (0.35–0.55)"),
        Patch(color=FRAUD_COLORS["Green"],  label="Low (< 0.35)"),
    ]
    ax_net.legend(handles=legend_elements, fontsize=10, loc="upper left")
    ax_net.set_title(
        "Fraud Network Visualization — Largest Connected Component\n"
        "(Node size & color = anomaly score; edges = shared risk attributes)",
        fontsize=12, fontweight="bold"
    )
    ax_net.axis("off")
    plt.tight_layout()
    savefig("03_fraud_network_visualization.png", folder=NET_DIR)

# ── Figure 4: Fraud Alert Monitoring Dashboard ────────────────────────────
fig4, axes4 = plt.subplots(2, 2, figsize=(16, 12))
fig4.suptitle(
    "Fraud Alert Monitoring & Escalation Dashboard",
    fontsize=13, fontweight="bold", color=PAL["primary"]
)

# Alert detection rate over time
axes4[0,0].plot(range(len(monitor_df)),
                 monitor_df["fraud_detection_rate"] * 100,
                 color=PAL["success"], lw=2)
axes4[0,0].axhline(85, color=PAL["warning"], linestyle="--", lw=1.5,
                    label="Target ≥ 85%")
axes4[0,0].set_title("Fraud Detection Rate Over 90 Days")
axes4[0,0].set_ylabel("Detection Rate (%)")
axes4[0,0].legend(fontsize=9)
axes4[0,0].set_ylim(0, 110)

# Cumulative fraud loss
axes4[0,1].fill_between(range(len(monitor_df)),
                          monitor_df["cum_fraud_loss_cr"],
                          alpha=0.4, color=PAL["danger"])
axes4[0,1].plot(range(len(monitor_df)),
                 monitor_df["cum_fraud_loss_cr"],
                 color=PAL["danger"], lw=2)
axes4[0,1].set_title("Cumulative Fraud Loss Over 90 Days (₹ Cr)")
axes4[0,1].set_ylabel("Fraud Loss (₹ Cr)")

# Alert type breakdown
if "primary_fraud_type" in alerts_df.columns:
    alert_types = alerts_df["primary_fraud_type"].value_counts()
    axes4[1,0].bar(range(len(alert_types)), alert_types.values,
                    color=SEVERITY_PALETTE * 5)
    axes4[1,0].set_xticks(range(len(alert_types)))
    axes4[1,0].set_xticklabels(
        [t.replace(" ", "\n") for t in alert_types.index], fontsize=8
    )
    axes4[1,0].set_title("Alert Volume by Primary Fraud Type")
    axes4[1,0].set_ylabel("Number of Alerts")

# False positive rate over time
axes4[1,1].plot(range(len(monitor_df)),
                 monitor_df["false_positive_rate"] * 100,
                 color=PAL["warning"], lw=2)
axes4[1,1].axhline(5, color=PAL["danger"], linestyle="--", lw=1.5,
                    label="Max 5% FPR")
axes4[1,1].set_title("False Positive Rate Over Time (%)")
axes4[1,1].set_ylabel("FPR (%)")
axes4[1,1].legend(fontsize=9)

plt.tight_layout()
savefig("04_fraud_alert_monitoring.png")

log.info("Executive fraud intelligence dashboards complete.")
print(f"\n  ✅  4 executive dashboards saved to {DASH_DIR}")


══════════════════════════════════════════════════════════════════════
  STEP 16 & 17 — EXECUTIVE FRAUD INTELLIGENCE DASHBOARDS
══════════════════════════════════════════════════════════════════════
11:04:34 | INFO     |   Figure: 01_executive_fraud_intelligence_dashboard.png
11:04:35 | INFO     |   Figure: 02_behavioral_fraud_analytics.png
11:04:35 | INFO     |   Figure: 03_fraud_network_visualization.png
11:04:36 | INFO     |   Figure: 04_fraud_alert_monitoring.png
11:04:36 | INFO     | Executive fraud intelligence dashboards complete.

  ✅  4 executive dashboards saved to C:\Users\abhir\OneDrive\Desktop\iitg\fraud_detection\dashboards


In [20]:
# =============================================================================
# CELL 19 — STEP 18: Fairness & Ethical Fraud Analytics
# =============================================================================

section("STEP 18 — FAIRNESS & ETHICAL FRAUD ANALYTICS")

# ─────────────────────────────────────────────────────────────────────────
# FAIRNESS IN FRAUD DETECTION:
# Fraud systems can perpetuate or amplify discrimination if:
# • Thin-file populations (young, rural, first-time borrowers) are
#   disproportionately flagged as synthetic identity suspects
# • Low-income borrowers are flagged for behavioral anomalies that
#   reflect financial distress rather than fraud intent
# • Geographic proxies create indirect demographic discrimination
#
# We evaluate:
# • Disparate flagging rates across protected characteristics
# • Calibration of fraud scores across groups
# • False positive burden by group
# ─────────────────────────────────────────────────────────────────────────

FAIRNESS_GROUPS_FRAUD = [
    "employment_type",
    "device_type",
    "acquisition_channel",
]

fairness_results = []
print("  Fraud Detection Fairness Analysis:")

for group_col in FAIRNESS_GROUPS_FRAUD:
    if group_col not in fd.columns:
        continue

    grp = fd.groupby(group_col).agg(
        n                    = ("fraud_risk_score",  "count"),
        avg_fraud_score      = ("fraud_risk_score",  "mean"),
        pct_red_orange       = ("fraud_severity",
                                 lambda x: (x.isin(["Red","Orange"])).mean()),
        pct_synthetic_id     = ("synthetic_id_flag", "mean"),
        pct_app_fraud        = ("app_fraud_flag",    "mean"),
        default_rate         = ("default_flag",      "mean"),
    ).reset_index()

    best_flag_rate = grp["pct_red_orange"].max()
    grp["disparate_impact"] = (
        grp["pct_red_orange"] / max(best_flag_rate, 0.001)
    ).round(4)

    # False positive burden: flagged but NOT defaulted
    fp_burden = fd.groupby(group_col).apply(
        lambda x: (x["fraud_severity"].isin(["Red","Orange"]) &
                   (x["default_flag"] == 0)).mean()
    ).rename("fp_burden")
    grp = grp.merge(fp_burden.reset_index(), on=group_col, how="left")

    grp["fairness_flag"] = grp["disparate_impact"].apply(
        lambda d: "FAIL ❌" if d > 1.5 else "WARN ⚠️" if d > 1.2 else "PASS ✅"
    )

    grp.to_csv(
        os.path.join(GOV_DIR, f"fairness_fraud_{group_col}.csv"), index=False
    )
    fairness_results.append({"dimension": group_col, "groups": grp})

    print(f"\n  {group_col}:")
    print(grp[[
        group_col, "n", "avg_fraud_score",
        "pct_red_orange", "pct_synthetic_id",
        "fp_burden", "fairness_flag"
    ]].to_string(index=False))

    fails = grp[grp["fairness_flag"].str.contains("FAIL")]
    warns = grp[grp["fairness_flag"].str.contains("WARN")]
    if len(fails):
        print(f"  ❌  FAIL: {len(fails)} groups with high disparate impact (>1.5×)")
    elif len(warns):
        print(f"  ⚠️   WARN: {len(warns)} groups need review")
    else:
        print(f"  ✅  All groups within fairness tolerance")

# ── Ethical Fraud Recommendations ─────────────────────────────────────────
ETHICAL_FRAUD_RECOMMENDATIONS = [
    "Thin-file borrowers (credit history < 6m) should receive ENHANCED onboarding "
    "support, not automatic fraud flagging. Distinguish data absence from fraud.",

    "First-Time Borrowers show higher SIF scores due to thin file — "
    "apply human review rather than automated decline to reduce false positives.",

    "Feature Phone users may appear as device-overlap fraud risk, but this "
    "reflects shared devices in rural households, not fraud. Apply geographic context.",

    "Behavioral signals like nighttime transactions may reflect shift workers "
    "and should not be over-indexed in fraud scores without corroborating signals.",

    "Fraud false positives should be tracked by employment type and geography "
    "quarterly and reviewed by an Ethics Committee to prevent systematic bias.",

    "Any borrower flagged as fraud-risk retains the right to appeal within 30 days. "
    "Appeals must be processed by a human reviewer.",
]

with open(os.path.join(GOV_DIR, "ethical_fraud_recommendations.json"), "w") as f:
    json.dump(ETHICAL_FRAUD_RECOMMENDATIONS, f, indent=2)

print("\n  Ethical Fraud Recommendations:")
for i, rec in enumerate(ETHICAL_FRAUD_RECOMMENDATIONS, 1):
    print(f"  {i}. {rec[:100]}...")

log.info("Fairness and ethical fraud analytics complete.")
print("\n  ✅  Fairness analysis and ethics recommendations saved.")


══════════════════════════════════════════════════════════════════════
  STEP 18 — FAIRNESS & ETHICAL FRAUD ANALYTICS
══════════════════════════════════════════════════════════════════════
  Fraud Detection Fairness Analysis:

  employment_type:
    employment_type     n  avg_fraud_score  pct_red_orange  pct_synthetic_id  fp_burden fairness_flag
First-Time Borrower  3223         0.270076        0.000621          0.362395   0.000310        PASS ✅
         Gig Worker  3213         0.321113        0.012138          0.366947   0.010582        PASS ✅
           Salaried 10736         0.177662        0.000000          0.024031   0.000000        PASS ✅
      Self-Employed  5055         0.206544        0.000000          0.030663   0.000000        PASS ✅
          Sme Owner  2372         0.236709        0.000422          0.027825   0.000422        PASS ✅
  ✅  All groups within fairness tolerance

  device_type:
  device_type    n  avg_fraud_score  pct_red_orange  pct_synthetic_id  fp_burden fa

In [21]:
# =============================================================================
# CELL 20 — STEP 19: Production Fraud Detection Pipeline
# =============================================================================

section("STEP 19 — PRODUCTION FRAUD DETECTION PIPELINE")

FRAUD_PIPELINE_CODE = '''
"""
PRODUCTION FRAUD DETECTION PIPELINE — Module 10
Save as: iitg/fraud_detection/pipelines/fraud_pipeline.py
Usage: from fraud_pipeline import FraudScorer, score_borrower, generate_alert
"""
import json, joblib, warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

BASE_DIR  = r"C:\\Users\\abhir\\OneDrive\\Desktop\\iitg"
FD_DIR    = f"{BASE_DIR}/fraud_detection"
MOD_DIR   = f"{FD_DIR}/anomaly_models"

# Load artifacts
_iforest  = joblib.load(f"{MOD_DIR}/isolation_forest.pkl")
_scaler   = joblib.load(f"{MOD_DIR}/anomaly_scaler.pkl")
_features = joblib.load(f"{MOD_DIR}/anomaly_features.pkl")

try:
    _fraud_model = joblib.load(f"{MOD_DIR}/supervised_fraud_model.pkl")
    _sup_feats   = joblib.load(f"{MOD_DIR}/supervised_fraud_features.pkl")
    SUP_OK = True
except FileNotFoundError:
    SUP_OK = False

EMP_INCOME_MEDIANS = {
    "Salaried": 45000, "Self-Employed": 52000, "Gig Worker": 22000,
    "SME Owner": 85000, "First-Time Borrower": 18000,
}


def compute_fraud_features(borrower: dict) -> dict:
    """Compute fraud features from raw borrower attributes."""
    emp = borrower.get("employment_type", "Salaried")
    income = max(borrower.get("monthly_income", 30000), 1)
    hist_len = borrower.get("credit_history_length", 12)

    identity_consistency = float(np.clip(
        0.90 - (hist_len < 6) * 0.20 + (borrower.get("income_stability_score", 0.6) - 0.5) * 0.20,
        0.01, 1.0
    ))
    document_risk = float(np.clip(
        0.05 + (1 - borrower.get("income_stability_score", 0.6)) * 0.30
        + (hist_len < 3) * 0.25,
        0.0, 1.0
    ))
    synthetic_id_prob = float(np.clip(
        (hist_len < 6) * 0.40
        + (borrower.get("loan_amount", 100000) > income * 20) * 0.25
        + document_risk * 0.20,
        0.0, 1.0
    ))
    income_inflation = income / EMP_INCOME_MEDIANS.get(emp, 40000)
    app_fraud = float(np.clip(
        (income_inflation > 2.5) * 0.35
        + document_risk * 0.25
        + (1 - identity_consistency) * 0.25
        + synthetic_id_prob * 0.15,
        0.0, 1.0
    ))
    sv = borrower.get("spending_volatility_index", 0.3)
    fsi = borrower.get("financial_stress_index", 0.3)
    rci = float(np.clip(
        (borrower.get("loan_amount", 100000) > borrower.get("bank_balance_avg", 50000) * 3) * 0.3
        + sv * 0.4,
        0.0, 1.0
    ))
    behavioral_fraud = float(np.clip(
        0.25 * sv + 0.25 * rci + 0.20 * borrower.get("avg_delay_days", 0) / 45,
        0.0, 1.0
    ))

    return {
        "identity_consistency_score":  identity_consistency,
        "document_risk_score":         document_risk,
        "synthetic_identity_prob":     synthetic_id_prob,
        "app_fraud_score":             app_fraud,
        "behavioral_fraud_score":      behavioral_fraud,
        "rapid_cashout_indicator":     rci,
        "income_inflation_ratio":      income_inflation,
    }


class FraudScorer:
    """
    Production fraud scoring engine.
    Provides real-time fraud risk assessment for individual borrowers.
    """

    def __init__(self):
        self.iforest  = _iforest
        self.scaler   = _scaler
        self.features = _features

    def score(self, borrower: dict) -> dict:
        """Score a single borrower. Returns complete fraud risk assessment."""
        fraud_feats = compute_fraud_features(borrower)
        merged = {**borrower, **fraud_feats}

        # Feature vector
        x = np.array([merged.get(f, 0) for f in self.features]).reshape(1, -1)
        x_scaled = self.scaler.transform(x)

        # Anomaly score
        anomaly_score = float(-self.iforest.score_samples(x_scaled)[0])
        anomaly_score_n = float(np.clip(
            (anomaly_score + 0.5) / 1.0, 0, 1
        ))

        # Composite fraud score
        fraud_score = float(np.clip(
            0.30 * anomaly_score_n
            + 0.25 * fraud_feats["app_fraud_score"]
            + 0.20 * fraud_feats["behavioral_fraud_score"]
            + 0.15 * fraud_feats["synthetic_identity_prob"]
            + 0.10 * fraud_feats.get("rapid_cashout_indicator", 0),
            0, 1
        ))

        if fraud_score >= 0.75:   severity = "Red"
        elif fraud_score >= 0.55: severity = "Orange"
        elif fraud_score >= 0.35: severity = "Yellow"
        else:                     severity = "Green"

        action = {
            "Red":    "HOLD DISBURSEMENT — Senior fraud review within 24h",
            "Orange": "Enhanced due diligence within 48h",
            "Yellow": "Enhanced monitoring; review next repayment",
            "Green":  "Standard processing",
        }[severity]

        return {
            "fraud_risk_score":         round(fraud_score, 4),
            "fraud_severity":           severity,
            "anomaly_score":            round(anomaly_score_n, 4),
            "app_fraud_score":          round(fraud_feats["app_fraud_score"], 4),
            "synthetic_id_prob":        round(fraud_feats["synthetic_identity_prob"], 4),
            "behavioral_fraud_score":   round(fraud_feats["behavioral_fraud_score"], 4),
            "income_inflation_ratio":   round(fraud_feats["income_inflation_ratio"], 2),
            "recommended_action":       action,
        }


def score_borrower(borrower: dict) -> dict:
    """Convenience API — score a single borrower dict."""
    return FraudScorer().score(borrower)


def generate_alert(scoring_result: dict, loan_id: str, cust_id: str) -> dict:
    """Generate a fraud alert record from a scoring result."""
    from datetime import datetime
    return {
        "alert_id":          f"FALERT_{loan_id}",
        "timestamp":         datetime.now().isoformat(),
        "loan_id":           loan_id,
        "customer_id":       cust_id,
        "fraud_risk_score":  scoring_result["fraud_risk_score"],
        "fraud_severity":    scoring_result["fraud_severity"],
        "recommended_action":scoring_result["recommended_action"],
        "status":            "Open",
    }


if __name__ == "__main__":
    test_borrower = {
        "credit_score": 540, "monthly_income": 95000,
        "loan_amount": 2000000, "employment_type": "Self-Employed",
        "credit_history_length": 3, "income_stability_score": 0.35,
        "financial_stress_index": 0.72, "spending_volatility_index": 0.68,
        "avg_delay_days": 28, "bank_balance_avg": 8000,
    }
    result = score_borrower(test_borrower)
    print(f"Fraud Risk Score: {result['fraud_risk_score']:.3f}")
    print(f"Severity        : {result['fraud_severity']}")
    print(f"Action          : {result['recommended_action']}")
    alert = generate_alert(result, "LOAN12345678", "CUST0001234")
    print(f"Alert generated : {alert['alert_id']}")
'''

with open(os.path.join(PIP_DIR, "fraud_pipeline.py"), "w", encoding="utf-8") as f:
    f.write(FRAUD_PIPELINE_CODE)

# ── Save final enriched portfolio with fraud scores ────────────────────────
fraud_cols = [
    "fraud_risk_score", "fraud_risk_score_100", "fraud_severity",
    "investigation_priority", "ensemble_anomaly_score",
    "app_fraud_score", "synthetic_id_score", "behavioral_fraud_score",
    "transaction_fraud_score", "behavioral_fraud_persona",
    "behavioral_fraud_level", "synthetic_id_level",
    "synthetic_id_flag", "app_fraud_flag", "mule_activity_flag",
    "ensemble_anomaly_flag", "repayment_manipulation_flag",
]
fraud_output_cols = [
    c for c in [
        "loan_id", "customer_id", "loan_amount", "employment_type",
        "city", "default_flag"
    ] + fraud_cols if c in fd.columns
]
fd[fraud_output_cols].to_csv(
    os.path.join(RPT_DIR, "portfolio_fraud_scores.csv"), index=False
)

log.info("Production fraud pipeline saved.")
print("  ✅  Production pipeline: fraud_detection/pipelines/fraud_pipeline.py")


══════════════════════════════════════════════════════════════════════
  STEP 19 — PRODUCTION FRAUD DETECTION PIPELINE
══════════════════════════════════════════════════════════════════════
11:04:53 | INFO     | Production fraud pipeline saved.
  ✅  Production pipeline: fraud_detection/pipelines/fraud_pipeline.py


In [22]:
# =============================================================================
# CELL 21 — Executive Summary & Module Orchestrator
# =============================================================================

section("MODULE 10 — EXECUTIVE SUMMARY")

n_red_alerts    = (fd["fraud_severity"] == "Red").sum()
n_orange_alerts = (fd["fraud_severity"] == "Orange").sum()
total_exposure  = fd[fd["fraud_severity"].isin(["Red","Orange"])]["loan_amount"].sum() / 1e7
sif_count       = fd["synthetic_id_flag"].sum()

EXEC_SUMMARY_M10 = f"""
╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 10 — FRAUD DETECTION & FINANCIAL CRIME INTELLIGENCE          ║
║  EXECUTIVE SUMMARY                                                   ║
║  Prepared for: CRO / Chief Compliance Officer / Fraud Analytics Head ║
║  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PORTFOLIO FRAUD INTELLIGENCE SNAPSHOT:                              ║
║  Total Loans Scored          : {len(fd):,}                              ║
║  Critical Alerts (Red)       : {n_red_alerts:,} ({n_red_alerts/len(fd):.2%})                   ║
║  High Alerts (Orange)        : {n_orange_alerts:,} ({n_orange_alerts/len(fd):.2%})               ║
║  Fraud Exposure (Red+Orange) : ₹{total_exposure:.2f} Cr                    ║
║  Synthetic ID Suspects       : {sif_count:,} ({sif_count/len(fd):.2%})                   ║
║  Mule Activity Flags         : {fd['mule_activity_flag'].sum():,}                              ║
║                                                                      ║
║  ANOMALY DETECTION SUMMARY:                                          ║
║  Isolation Forest Anomalies  : {fd['iforest_flag'].sum():,} ({fd['iforest_flag'].mean():.1%})        ║
║  LOF Anomalies               : {fd['lof_flag'].sum():,} ({fd['lof_flag'].mean():.1%})        ║
║  Ensemble Anomalies (P95+)   : {fd['ensemble_anomaly_flag'].sum():,} ({fd['ensemble_anomaly_flag'].mean():.1%})        ║
║                                                                      ║
║  NETWORK FRAUD INTELLIGENCE:                                         ║
║  Network nodes (borrowers)   : {G.number_of_nodes():,}                              ║
║  Network edges (connections) : {G.number_of_edges():,}                              ║
║                                                                      ║
║  STRESS RESILIENCE:                                                  ║
║  Adversarial scenarios tested: 5                                     ║
║  Worst-case degradation       : See stress testing report            ║
║                                                                      ║
║  TOP STRATEGIC PRIORITIES:                                           ║
║  1. Review all {n_red_alerts:,} Critical (Red) borrowers within 24h         ║
║  2. Investigate synthetic ID cluster — {sif_count:,} suspects           ║
║  3. Deploy real-time fraud pipeline (fraud_pipeline.py)              ║
║  4. Implement monthly fairness review for false positive burden      ║
║  5. Schedule quarterly adversarial robustness testing               ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(EXEC_SUMMARY_M10)

with open(os.path.join(RPT_DIR, "MODULE10_EXECUTIVE_SUMMARY.txt"),
           "w", encoding="utf-8") as f:
    f.write(EXEC_SUMMARY_M10)

# ── Output inventory ───────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  MODULE 10 — OUTPUT INVENTORY")
print("═" * 70)

output_inventory = {
    "🤖 Anomaly Models": [
        "isolation_forest.pkl       — Isolation Forest anomaly detector",
        "one_class_svm.pkl          — One-Class SVM anomaly detector",
        "autoencoder.pkl            — Autoencoder reconstruction error detector",
        "supervised_fraud_model.pkl — LightGBM supervised fraud model",
        "anomaly_scaler.pkl         — Feature scaler",
    ],
    "🕸️  Fraud Networks": [
        "fraud_network.gexf          — Gephi-ready fraud network",
        "fraud_communities.csv       — Community detection results",
        "03_fraud_network_visualization.png — Network graph",
    ],
    "🔍 Explainability": [
        "fraud_shap_importance.csv   — SHAP feature importance",
        "top_fraud_explanations.json — Investigation-ready explanations",
    ],
    "🚨 Alerts": [
        "fraud_alert_queue.csv       — Investigation queue (all active alerts)",
        "fraud_alert_summary.csv     — Alert summary by severity",
        "synthetic_identity_alerts.csv — SIF investigation queue",
    ],
    "📋 Reports": [
        "fraud_features_portfolio.csv — Fraud feature table",
        "fraud_risk_ladder.csv        — Risk tier summary",
        "behavioral_fraud_personas.csv— Persona-level analytics",
        "app_fraud_by_employment.csv  — Application fraud breakdown",
        "transaction_fraud_heatmap.csv— Transaction fraud heatmap",
        "portfolio_fraud_scores.csv   — Full portfolio with fraud scores",
        "MODULE10_EXECUTIVE_SUMMARY.txt — CRO summary",
    ],
    "🏛️  Governance": [
        "fraud_governance_objectives.json  — Governance targets",
        "fraud_taxonomy.json               — Fraud category taxonomy",
        "fraud_severity_hierarchy.json     — Severity classification",
        "fraud_escalation_framework.json   — Investigation escalation",
        "realtime_pipeline_architecture.json — Production design",
        "realtime_fraud_monitoring.csv     — 90-day monitoring simulation",
        "fraud_governance_scorecard.csv    — Governance KPIs",
        "fraud_audit_trail.csv             — Audit-ready decision log",
        "ethical_fraud_recommendations.json— Ethics guidelines",
        "fairness_fraud_*.csv              — Fairness analysis by group",
    ],
    "🌡️  Stress Testing": [
        "adversarial_stress_results.csv — 5-scenario adversarial simulation",
    ],
    "🎛️  Dashboards": [
        "01_executive_fraud_intelligence_dashboard.png — CRO dashboard",
        "02_behavioral_fraud_analytics.png — Behavioral heatmaps",
        "03_fraud_network_visualization.png — Network graph",
        "04_fraud_alert_monitoring.png — Alert tracking dashboard",
    ],
    "🔧 Pipelines": [
        "fraud_pipeline.py           — Production fraud scoring API",
    ],
}

for category, items in output_inventory.items():
    print(f"\n  {category}:")
    for item in items:
        print(f"    • {item}")

print(f"\n  Output Directory: {FD_DIR}")
print("═" * 70)
log.info("Module 10 — Fraud Detection & Financial Crime Intelligence complete.")


def get_module10_artefacts() -> dict:
    """
    Returns all Module 10 outputs for downstream modules (11–15).
    Key consumers:
      Module 11 → fraud_ladder, alerts_df, exec summary
      Module 12 → fraud_pipeline.py, fraud scoring API
      Module 14 → adversarial stress, monitoring simulation
      Module 15 → governance scorecard, ethical recommendations
    """
    return {
        "fd":                  fd,
        "fraud_ladder":        fraud_ladder,
        "alerts_df":           alerts_df,
        "monitor_df":          monitor_df,
        "stress_df":           stress_df,
        "G":                   G,
        "FRAUD_TAXONOMY":      FRAUD_TAXONOMY,
        "FRAUD_SEVERITY_HIERARCHY": FRAUD_SEVERITY_HIERARCHY,
        "ESCALATION_FRAMEWORK":ESCALATION_FRAMEWORK,
        "gov_scorecard":       gov_scorecard,
        "EXEC_SUMMARY_M10":    EXEC_SUMMARY_M10,
        "FD_DIR":              FD_DIR,
        "MOD_DIR":             MOD_DIR,
        "NET_DIR":             NET_DIR,
        "ALERT_DIR":           ALERT_DIR,
        "RPT_DIR":             RPT_DIR,
        "GOV_DIR":             GOV_DIR,
        "PIP_DIR":             PIP_DIR,
    }

print("\n✅  Module 10 orchestrator ready. Call get_module10_artefacts() to export.")


══════════════════════════════════════════════════════════════════════
  MODULE 10 — EXECUTIVE SUMMARY
══════════════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════════════════════╗
║  MODULE 10 — FRAUD DETECTION & FINANCIAL CRIME INTELLIGENCE          ║
║  EXECUTIVE SUMMARY                                                   ║
║  Prepared for: CRO / Chief Compliance Officer / Fraud Analytics Head ║
║  Generated: 2026-05-23 11:04                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  PORTFOLIO FRAUD INTELLIGENCE SNAPSHOT:                              ║
║  Total Loans Scored          : 24,599                              ║
║  Critical Alerts (Red)       : 0 (0.00%)                   ║
║  High Alerts (Orange)        : 42 (0.17%)               ║
║  Fraud Exposure (Red+Orange) : ₹2.08 Cr                